# E-Commerce Data Warehouse Pipeline

This notebook implements a complete data warehouse pipeline for the Udacity Data Engineering course.

**Business Context:** You are a data engineer at a fast-growing e-commerce company. Critical data is spread across multiple operational systems (PostgreSQL, Cassandra, Neo4j), making it difficult for analysts to run consistent reports. Your job is to design and implement a centralized analytics warehouse in Amazon Redshift.

## Tasks Overview

1. **Explore and Plan** - Review CSV data, identify key fields, map to warehouse schema
2. **Design the Schema** - Create dimensional model with staging, dimension, and fact tables
3. **Extract and Transform** - Load source systems and extract/transform data
4. **Load into Redshift** - Execute DDL, load staging tables, populate dimensions and facts
5. **Optimize Performance** - Apply best practices, create materialized views
6. **Validate and Report** - Run quality checks, generate final report

---
## The Data

The dataset consists of three CSV files representing data from different operational systems:

### Orders Data (PostgreSQL source) - `ecom_orders_postgres.csv`
- **order_id**: Unique identifier for each order
- **customer_id**: Customer who placed the order
- **order_datetime / ship_datetime**: Timestamps for order and shipping
- **channel / device_type / browser**: How the order was placed
- **country / state**: Geographic location
- **payment_method / campaign**: Payment and marketing info
- **Financial fields**: subtotal, discount, shipping, tax, total amounts
- **Delivery fields**: delivery_days, on_time_delivery
- **Flags**: authorization_approved, returned

### Events Data (Cassandra source) - `ecom_events_cassandra.csv`
- **event_id / session_id**: Event and session identifiers
- **customer_id**: Customer who triggered the event
- **event_type**: Type of event (page_view, product_view, add_to_cart, etc.)
- **event_ts**: Timestamp of the event
- **Device/browser/OS info**: Technical context
- **Behavioral fields**: page_depth, latency_ms, dwell_seconds
- **Commerce fields**: cart_value_usd, discount_rate, fraud_score

### Graph Edges Data (Neo4j source) - `ecom_graph_edges_neo4j.csv`
- **edge_id**: Unique relationship identifier
- **from_node_id / to_node_id**: Source and target nodes
- **from_node_type / to_node_type**: Node types (Customer, Product, Order)
- **relationship**: Type of relationship (PURCHASED, VIEWED, ADDED_TO_CART, etc.)
- **Context fields**: order_id, category, customer_segment, campaign
- **Metrics**: edge_strength, unit_price_usd, quantity

---
## Setup: Imports and Dependencies

Run this cell first to import all required libraries.

In [2]:
# ========= Imports
import os, io, re, time, json, textwrap
from datetime import datetime
from typing import Dict, Any, List, Tuple
import numpy as np
import pandas as pd

# Source system libraries
import psycopg2
from psycopg2.extras import execute_values
from sqlalchemy import create_engine
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from neo4j import GraphDatabase

# Warehouse (Redshift) via Data API
import boto3

# Optional progress bars
try:
    from tqdm import tqdm
    TQDM = True
except Exception:
    TQDM = False

print("All imports successful!")
print(f"   - pandas version: {pd.__version__}")
print(f"   - numpy version: {np.__version__}")

All imports successful!
   - pandas version: 3.0.6
   - numpy version: 2.5.3


---
## Setup: Configuration

Update these settings for your environment. You will need to:
1. Set your AWS credentials (from Cloud Resources)
2. Configure database connection parameters

In [5]:
# Credentials and connection settings are read from .env (see .env.example).
from dotenv import load_dotenv
load_dotenv(override=True)


True

In [6]:
# ========= Configuration
BASE_DIR = os.getenv("PROJECT_BASE_DIR", ".")
DATA_DIR = os.path.join(BASE_DIR, "data")
CSV_ORDERS  = os.path.join(DATA_DIR, "ecom_orders_postgres.csv")
CSV_EVENTS  = os.path.join(DATA_DIR, "ecom_events_cassandra.csv")
CSV_EDGES   = os.path.join(DATA_DIR, "ecom_graph_edges_neo4j.csv")
DDL_MD_PATH = os.path.join(BASE_DIR, "project-ddl-long.md")
MERMAID_MD  = os.path.join(BASE_DIR, "project-mermaid-diagram.md")
BATCH_SIZE  = int(os.getenv("BATCH_SIZE", "1000"))

# PostgreSQL
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB",   "postgres")
PG_USER = os.getenv("PG_USER", "temp")
PG_PW   = os.getenv("PG_PW",   "temp")

# Cassandra
CAS_HOSTS = os.getenv("CAS_HOSTS", "localhost").split(",")
CAS_PORT  = int(os.getenv("CAS_PORT", "9042"))
CAS_USER  = os.getenv("CAS_USER", "")
CAS_PW    = os.getenv("CAS_PW", "")
CAS_KEYSPACE = os.getenv("CAS_KEYSPACE", "ecommerce")

# Neo4j
NEO4J_URI  = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PW   = os.getenv("NEO4J_PW",   "neo4jpass")

# AWS/Redshift
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")
REDSHIFT_DATABASE = os.getenv("REDSHIFT_DATABASE", "ecom")
REDSHIFT_WORKGROUP = os.getenv("REDSHIFT_WORKGROUP", "udacity-dwh-wg")
REDSHIFT_SECRET_ARN = os.getenv("REDSHIFT_SECRET_ARN")
REDSHIFT_CLUSTER_IDENTIFIER = os.getenv("REDSHIFT_CLUSTER_IDENTIFIER")
REDSHIFT_DB_USER = os.getenv("REDSHIFT_DB_USER")

# Verify configuration
print("Configuration loaded!")
print(f"   - BASE_DIR: {BASE_DIR}")
print(f"   - PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"   - Cassandra: {CAS_HOSTS}:{CAS_PORT}/{CAS_KEYSPACE}")
print(f"   - Neo4j: {NEO4J_URI}")
print(f"   - Redshift: {REDSHIFT_DATABASE} (workgroup: {REDSHIFT_WORKGROUP})")

print()
if AWS_ACCESS_KEY_ID and AWS_SECRET_ACCESS_KEY:
    print(f"   AWS credentials found (Key ID: {AWS_ACCESS_KEY_ID[:10]}...)")
else:
    print("   WARNING: AWS credentials NOT FOUND - set them above!")

Configuration loaded!
   - BASE_DIR: .
   - PostgreSQL: localhost:5432/postgres
   - Cassandra: ['localhost']:9042/ecommerce
   - Neo4j: bolt://localhost:7687
   - Redshift: ecom (workgroup: udacity-dwh-wg)

   AWS credentials found (Key ID: ASIAQ6JP6R...)


---
## Setup: Column Specifications

These define the mapping from source columns to Redshift staging tables.
Use these as a reference when building your transformation logic.

In [45]:
# Column specs for Redshift staging (name, kind)
# kind: 's' = string, 'ts' = timestamp, 'i' = integer, 'f' = float, 'b' = boolean

ORDERS_COLSPEC = [
    ('order_id','s'),('customer_id','s'),('order_datetime','ts'),('ship_datetime','ts'),
    ('channel','s'),('device_type','s'),('browser','s'),('country','s'),('state','s'),
    ('payment_method','s'),('campaign','s'),('primary_category','s'),('num_distinct_items','i'),
    ('subtotal_usd','f'),('discount_rate','f'),('discount_amount_usd','f'),('shipping_method','s'),
    ('shipping_cost_usd','f'),('tax_rate','f'),('tax_amount_usd','f'),('order_total_usd','f'),
    ('order_weight_kg','f'),('delivery_days','i'),('on_time_delivery','b'),
    ('authorization_approved','b'),('returned','b')
]

EVENTS_COLSPEC = [
    ('event_id','s'),('customer_id','s'),('session_id','s'),('event_type','s'),('event_ts','ts'),
    ('device_type','s'),('browser','s'),('os','s'),('referrer','s'),('country','s'),('state','s'),
    ('ab_variant','s'),('is_logged_in','b'),('page_depth','i'),('latency_ms','i'),
    ('dwell_seconds','i'),('cart_value_usd','f'),('discount_rate','f'),('fraud_score','f'),
    ('payment_outcome','s'),('sequence_num','i'),('product_id','s'),('category','s'),('promo_code','s')
]

EDGES_COLSPEC = [
    ('edge_id','s'),('from_node_id','s'),('from_node_type','s'),('to_node_id','s'),('to_node_type','s'),
    ('relationship','s'),('timestamp','ts'),('order_id','s'),('category','s'),('customer_segment','s'),
    ('edge_strength','f'),('price_bucket','s'),('region','s'),('state','s'),('campaign','s'),
    ('same_household','b'),('prior_interactions','i'),('dwell_seconds','i'),('product_id','s'),
    ('unit_price_usd','f'),('quantity','i'),('returned_flag','b'),('auth_approved','b')
]

print(f"Column specs defined:")
print(f"   - Orders: {len(ORDERS_COLSPEC)} columns")
print(f"   - Events: {len(EVENTS_COLSPEC)} columns")
print(f"   - Edges: {len(EDGES_COLSPEC)} columns")

Column specs defined:
   - Orders: 26 columns
   - Events: 24 columns
   - Edges: 23 columns


---
## Setup: Helper Functions

Utility functions used throughout the pipeline.

In [46]:
def trim_df(df: pd.DataFrame) -> pd.DataFrame:
    """Standardize text fields and handle NaN values."""
    df = df.copy()
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].astype(str).str.strip()
        df[c] = df[c].replace({'nan': np.nan, 'None': np.nan, 'NaN': np.nan, '': np.nan})
    return df

def read_csvs() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Read all three source CSVs and apply cleaning."""
    orders = pd.read_csv(CSV_ORDERS, keep_default_na=False, na_values=[''])
    events = pd.read_csv(CSV_EVENTS, keep_default_na=False, na_values=[''])
    edges  = pd.read_csv(CSV_EDGES, keep_default_na=False, na_values=[''])
    return trim_df(orders), trim_df(events), trim_df(edges)

print("Helper functions defined: trim_df(), read_csvs()")

Helper functions defined: trim_df(), read_csvs()


---
# Task 1: Explore and Plan the Data Pipeline

In this task, you will:
- Review the provided CSV data files to understand structure, columns, and content
- Identify key fields and relationships important for analysis
- Map source fields to fact and dimension tables
- Consider data format standardization needs

**Deliverables:**
- Written plan mapping fields from all three sources to fact and dimension tables
- Documentation of key relationships and ID standardization strategies

In [47]:
# Read the CSV files
print("Reading CSV files...")
orders_df, events_df, edges_df = read_csvs()

print("\n" + "="*60)
print("TASK 1: Data Exploration")
print("="*60)

# Explore the orders data
print(f"\n📊 ORDERS DATA (from PostgreSQL)")
print(f"   Shape: {orders_df.shape[0]} rows, {orders_df.shape[1]} columns")
print(f"   Columns: {list(orders_df.columns)}")
print("\n   Dtypes:")
print(orders_df.dtypes.to_string())
display(orders_df.head())


Reading CSV files...

TASK 1: Data Exploration

📊 ORDERS DATA (from PostgreSQL)
   Shape: 2500 rows, 26 columns
   Columns: ['order_id', 'customer_id', 'order_datetime', 'ship_datetime', 'channel', 'device_type', 'browser', 'country', 'state', 'payment_method', 'campaign', 'primary_category', 'num_distinct_items', 'subtotal_usd', 'discount_rate', 'discount_amount_usd', 'shipping_method', 'shipping_cost_usd', 'tax_rate', 'tax_amount_usd', 'order_total_usd', 'order_weight_kg', 'delivery_days', 'on_time_delivery', 'authorization_approved', 'returned']

   Dtypes:
order_id                      str
customer_id                   str
order_datetime                str
ship_datetime                 str
channel                       str
device_type                   str
browser                       str
country                       str
state                         str
payment_method                str
campaign                      str
primary_category              str
num_distinct_items       

/var/folders/sh/tl74dmfn2d38g0d22l4xvvhw0000gn/T/ipykernel_5181/3068392630.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in df.select_dtypes(include=['object']).columns:
/var/folders/sh/tl74dmfn2d38g0d22l4xvvhw0000gn/T/ipykernel_5181/3068392630.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas

,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
0,ORD100000,C29457,2024-06-12 02:14:21,2024-06-17 02:14:21,android_app,mobile,Safari,CA,CT,google_pay,...,standard,5.05,0.0000,0.00,838.39,0.10,5,True,True,False
1,ORD100001,C22666,2024-08-08 00:16:41,2024-08-11 00:16:41,mobile_web,desktop,Opera,DE,OH,apple_pay,...,standard,9.35,0.0000,0.00,388.60,0.20,3,True,True,False
2,ORD100002,C72623,2024-09-11 06:59:18,2024-09-15 06:59:18,web,desktop,Edge,US,TX,google_pay,...,standard,6.63,0.0923,59.32,708.60,0.85,4,True,True,False
3,ORD100003,C62733,2025-03-13 15:54:14,2025-03-17 15:54:14,web,mobile,Safari,UK,VT,google_pay,...,standard,7.15,0.0000,0.00,104.23,0.10,4,True,True,False
4,ORD100004,C62083,2025-02-28 00:20:21,2025-03-06 00:20:21,android_app,tablet,Opera,US,TN,apple_pay,...,standard,3.96,0.0864,3.69,50.41,0.29,6,False,True,False


In [48]:
# Explore the events data
print(f"\n📊 EVENTS DATA (from Cassandra)")
print(f"   Shape: {events_df.shape[0]} rows, {events_df.shape[1]} columns")
print(f"   Columns: {list(events_df.columns)}")
print("\n   Event types:")
print(events_df["event_type"].value_counts().to_string())
display(events_df.head())



📊 EVENTS DATA (from Cassandra)
   Shape: 2500 rows, 24 columns
   Columns: ['event_id', 'customer_id', 'session_id', 'event_type', 'event_ts', 'device_type', 'browser', 'os', 'referrer', 'country', 'state', 'ab_variant', 'is_logged_in', 'page_depth', 'latency_ms', 'dwell_seconds', 'cart_value_usd', 'discount_rate', 'fraud_score', 'payment_outcome', 'sequence_num', 'product_id', 'category', 'promo_code']

   Event types:
event_type
product_view        696
page_view           641
add_to_cart         480
checkout_start      259
payment_attempt     218
purchase            155
return_initiated     51


,event_id,customer_id,session_id,event_type,event_ts,device_type,browser,os,referrer,country,...,latency_ms,dwell_seconds,cart_value_usd,discount_rate,fraud_score,payment_outcome,sequence_num,product_id,category,promo_code
0,EVT200000,C47857,S7645111197,page_view,2024-11-14 00:17:09,desktop,Firefox,Linux,organic_search,IN,...,331,35,18.12,0.165,0.000,NaN,3,NaN,NaN,NONE
1,EVT200001,C83195,S1423573902,page_view,2024-08-15 10:42:18,mobile,Chrome,Linux,social,DE,...,432,26,33.62,0.083,0.262,NaN,2,NaN,NaN,NONE
2,EVT200002,C27664,S2829853742,product_view,2024-05-03 23:28:24,mobile,Chrome,Android,email,BR,...,360,6,12.26,0.237,0.162,NaN,5,P5735,Home,FREESHIP
3,EVT200003,C55911,S2500554740,add_to_cart,2024-10-28 18:44:50,desktop,Firefox,Linux,direct,US,...,414,22,86.55,0.197,0.067,NaN,1,P1730,Pets,FREESHIP
4,EVT200004,C27347,S3072616664,add_to_cart,2024-07-22 07:43:54,mobile,Chrome,Linux,direct,CA,...,381,32,87.78,0.058,0.000,NaN,8,P6252,Pets,WELCOME10


In [49]:
# Explore the graph edges data
print(f"\n📊 GRAPH EDGES DATA (from Neo4j)")
print(f"   Shape: {edges_df.shape[0]} rows, {edges_df.shape[1]} columns")
print(f"   Columns: {list(edges_df.columns)}")
print("\n   Relationships by node types:")
print(edges_df.groupby(["from_node_type", "relationship", "to_node_type"]).size().to_string())
display(edges_df.head())



📊 GRAPH EDGES DATA (from Neo4j)
   Shape: 2500 rows, 23 columns
   Columns: ['edge_id', 'from_node_id', 'from_node_type', 'to_node_id', 'to_node_type', 'relationship', 'timestamp', 'order_id', 'category', 'customer_segment', 'edge_strength', 'price_bucket', 'region', 'state', 'campaign', 'same_household', 'prior_interactions', 'dwell_seconds', 'product_id', 'unit_price_usd', 'quantity', 'returned_flag', 'auth_approved']

   Relationships by node types:
from_node_type  relationship      to_node_type
Customer        ADDED_TO_CART     Product         441
                PURCHASED         Product         469
                REFERRED_FRIEND   Customer        256
                RETURNS           Product         111
                VIEWED            Product         830
Product         ALSO_BOUGHT_WITH  Product         393


,edge_id,from_node_id,from_node_type,to_node_id,to_node_type,relationship,timestamp,order_id,category,customer_segment,...,state,campaign,same_household,prior_interactions,dwell_seconds,product_id,unit_price_usd,quantity,returned_flag,auth_approved
0,EDGE300000,C65482,Customer,P2261,Product,RETURNS,2024-01-27 13:42:26,ORD100938,Books,New,...,WA,SpringSale,False,3,66,P2261,315.87,2,False,False
1,EDGE300001,C59599,Customer,P7625,Product,PURCHASED,2025-02-07 10:22:55,ORD102431,Toys,Loyal,...,WI,Holiday,False,4,62,P7625,19.21,1,False,True
2,EDGE300002,C44719,Customer,P8677,Product,ADDED_TO_CART,2024-01-06 14:04:37,NaN,Pets,Active,...,KS,Loyalty,False,5,16,P8677,68.68,2,False,False
3,EDGE300003,C13897,Customer,P7883,Product,VIEWED,2024-06-04 21:36:14,NaN,Grocery,Active,...,NV,Clearance,False,3,44,P7883,74.70,1,False,False
4,EDGE300004,C30350,Customer,P2015,Product,ADDED_TO_CART,2024-06-23 20:00:17,NaN,Toys,Active,...,AR,SpringSale,False,2,17,P2015,52.71,1,False,False


In [50]:
# Identify key fields and relationships
print("\n" + "="*60)
print("KEY FIELDS AND RELATIONSHIPS")
print("="*60)

SOURCES = [
    ("orders", orders_df, ORDERS_COLSPEC, "order_id"),
    ("events", events_df, EVENTS_COLSPEC, "event_id"),
    ("edges",  edges_df,  EDGES_COLSPEC,  "edge_id"),
]

print("\n Primary Keys:")
for name, df, _, key in SOURCES:
    row_count = len(df)
    unique_count = df[key].nunique()
    status = "unique" if unique_count == row_count else "DUPLICATES"
    print(f"   {name}.{key}: {unique_count:,} unique of {row_count:,} rows -> {status}")

print("\n Foreign Key Relationships:")
cust_orders = set(orders_df.customer_id.dropna())
cust_events = set(events_df.customer_id.dropna())
cust_edges = (set(edges_df.loc[edges_df.from_node_type == "Customer", "from_node_id"])
              | set(edges_df.loc[edges_df.to_node_type == "Customer", "to_node_id"]))
print(f"   customers: orders={len(cust_orders):,}, events={len(cust_events):,}, edges={len(cust_edges):,}")
print(f"   orders ∩ events: {len(cust_orders & cust_events):,}   orders ∩ edges: {len(cust_orders & cust_edges):,}   "
      f"all sources (union): {len(cust_orders | cust_events | cust_edges):,}")
prod_events = set(events_df.product_id.dropna())
prod_edges = (set(edges_df.product_id.dropna())
              | set(edges_df.loc[edges_df.from_node_type == "Product", "from_node_id"])
              | set(edges_df.loc[edges_df.to_node_type == "Product", "to_node_id"]))
print(f"   products: events={len(prod_events):,}, edges={len(prod_edges):,}, overlap={len(prod_events & prod_edges):,}")
edge_orders = set(edges_df.order_id.dropna())
print(f"   edges.order_id found in orders: {len(edge_orders & set(orders_df.order_id)):,} of {len(edge_orders):,}")
sess = events_df.session_id.nunique()
print(f"   events: {sess:,} sessions across {len(events_df):,} events")

print("\n Date/Time Fields:")
for source_name, source_df, colspec, _primary_key in SOURCES:
    for column, kind in colspec:
        if kind == "ts":
            timestamps = pd.to_datetime(source_df[column])
            print(f"   {source_name}.{column}: {timestamps.min()} -> {timestamps.max()}  (nulls: {timestamps.isna().sum()})")

print("\n Categorical Fields (potential dimensions):")
for source_name, source_df, colspec, _primary_key in SOURCES:
    categorical_cardinality = {
        column: source_df[column].nunique()
        for column, kind in colspec
        if kind == "s" and not column.endswith("_id") and source_df[column].nunique() <= 50
    }
    print(f"   {source_name}: {categorical_cardinality}")


KEY FIELDS AND RELATIONSHIPS

 Primary Keys:
   orders.order_id: 2,500 unique of 2,500 rows -> unique
   events.event_id: 2,500 unique of 2,500 rows -> unique
   edges.edge_id: 2,500 unique of 2,500 rows -> unique

 Foreign Key Relationships:
   customers: orders=2,460, events=2,461, edges=2,340
   orders ∩ events: 72   orders ∩ edges: 72   all sources (union): 7,057
   products: events=1,276, edges=2,462, overlap=378
   edges.order_id found in orders: 796 of 796
   events: 2,500 sessions across 2,500 events

 Date/Time Fields:
   orders.order_datetime: 2024-01-01 01:48:15 -> 2025-06-29 15:18:02  (nulls: 0)
   orders.ship_datetime: 2024-01-03 21:57:58 -> 2025-07-06 04:31:14  (nulls: 0)
   events.event_ts: 2024-01-01 09:05:14 -> 2025-06-29 19:22:01  (nulls: 0)
   edges.timestamp: 2024-01-01 01:19:07 -> 2025-06-29 22:30:45  (nulls: 0)

 Categorical Fields (potential dimensions):
   orders: {'channel': 5, 'device_type': 3, 'browser': 5, 'country': 6, 'payment_method': 5, 'campaign': 6, '

In [51]:
# Document your field mappings and data quality observations
print("\n" + "="*60)
print("DATA QUALITY SUMMARY")
print("="*60)

# Null values per source (only columns that have any)
for name, df, _, key in SOURCES:
    nulls = df.isna().sum()
    nulls = nulls[nulls > 0]
    print(f"\n{name}: {len(df):,} rows, null key ({key}) = {df[key].isna().sum()}, duplicate keys = {df[key].duplicated().sum()}")
    if nulls.empty:
        print("   no null values")
    else:
        print((nulls.to_frame("nulls").assign(pct=lambda x: (x.nulls / len(df) * 100).round(1))).to_string())

# Business-rule checks on orders
calc = orders_df.subtotal_usd - orders_df.discount_amount_usd + orders_df.shipping_cost_usd + orders_df.tax_amount_usd
print(f"\norders where total != subtotal - discount + shipping + tax: {((calc - orders_df.order_total_usd).abs() > 0.01).sum():,}")
print(f"orders shipped before ordered: {(pd.to_datetime(orders_df.ship_datetime) < pd.to_datetime(orders_df.order_datetime)).sum():,}")
print(f"events with negative latency/dwell: {((events_df.latency_ms < 0) | (events_df.dwell_seconds < 0)).sum():,}")
print(f"edges with null from/to node id: {edges_df[['from_node_id', 'to_node_id']].isna().any(axis=1).sum():,}")



DATA QUALITY SUMMARY

orders: 2,500 rows, null key (order_id) = 0, duplicate keys = 0
          nulls   pct
campaign    409  16.4

events: 2,500 rows, null key (event_id) = 0, duplicate keys = 0
                 nulls   pct
state             2210  88.4
payment_outcome   2127  85.1
product_id        1118  44.7
category          1118  44.7

edges: 2,500 rows, null key (edge_id) = 0, duplicate keys = 0
          nulls   pct
order_id   1527  61.1
campaign    352  14.1

orders where total != subtotal - discount + shipping + tax: 0
orders shipped before ordered: 0
events with negative latency/dwell: 0
edges with null from/to node id: 0


## Field Mapping: Source Fields → Dimensional Model

Every source field (orders 26, events 24, edges 23) mapped to the warehouse table it lands in. Surrogate-key lookups join on `is_current = TRUE` for `dim_customer` and `dim_product`; date keys are derived as `yyyymmdd`.

### PostgreSQL `raw.orders` → `stg_orders_raw`

| Source field | Target table | Target column | Mapping |
|---|---|---|---|
| `order_id` | `fact_orders` | `order_id` | Degenerate key (grain) |
| `customer_id` | `dim_customer` → `fact_orders` | `customer_id` → `customer_sk` | Business key → surrogate key |
| `country`, `state` | `dim_customer` | `country`, `state` | Customer attribute |
| `order_datetime` | `dim_date` → `fact_orders` | `date_key` → `order_date_key` | Derived `yyyymmdd` |
| `ship_datetime` | `dim_date` → `fact_orders` | `date_key` → `ship_date_key` | Derived `yyyymmdd` |
| `channel` | `dim_channel` → `fact_orders` | `channel_sk` | Lookup dimension |
| `device_type` | `dim_device` → `fact_orders` | `device_sk` | Lookup dimension |
| `browser` | `dim_browser` → `fact_orders` | `browser_sk` | Lookup dimension |
| `campaign` | `dim_campaign` → `fact_orders` | `campaign_sk` | Lookup dimension |
| `payment_method` | `dim_payment_method` → `fact_orders` | `payment_method_sk` | Lookup dimension |
| `shipping_method` | `dim_shipping_method` → `fact_orders` | `shipping_method_sk` | Lookup dimension |
| `primary_category`, `num_distinct_items`, `subtotal_usd`, `discount_rate`, `discount_amount_usd`, `shipping_cost_usd`, `tax_rate`, `tax_amount_usd`, `order_total_usd`, `order_weight_kg`, `delivery_days`, `on_time_delivery`, `authorization_approved`, `returned` | `fact_orders` | same name | Measure / attribute, loaded as-is |

### Cassandra `ecommerce.events` → `stg_events_raw`

| Source field | Target table | Target column | Mapping |
|---|---|---|---|
| `event_id` | `fact_events` | `event_id` | Degenerate key (grain) |
| `customer_id` | `dim_customer` → `fact_events` | `customer_id` → `customer_sk` | Business key → surrogate key |
| `country`, `state`, `is_logged_in` | `dim_customer` | same name | Customer attribute |
| `product_id` | `dim_product` → `fact_events` | `product_id` → `product_sk` | Business key → surrogate key |
| `category` | `dim_product` and `fact_events` | `category` | Product attribute, also kept on the fact |
| `event_ts` | `dim_date` → `fact_events` | `date_key` → `event_date_key` | Derived `yyyymmdd` |
| `device_type` | `dim_device` → `fact_events` | `device_sk` | Lookup dimension |
| `browser` | `dim_browser` → `fact_events` | `browser_sk` | Lookup dimension |
| `os` | `dim_os` → `fact_events` | `os_sk` | Lookup dimension |
| `referrer` | `dim_referrer` → `fact_events` | `referrer_sk` | Lookup dimension |
| `ab_variant` | `dim_ab_variant` → `fact_events` | `ab_variant_sk` | Lookup dimension |
| `session_id`, `event_type` | `fact_events` | same name | Degenerate dimension |
| `page_depth`, `latency_ms`, `dwell_seconds`, `cart_value_usd`, `discount_rate`, `fraud_score`, `payment_outcome`, `sequence_num`, `promo_code` | `fact_events` | same name | Measure / attribute, loaded as-is |

### Neo4j edges → `stg_edges_raw`

| Source field | Target table | Target column | Mapping |
|---|---|---|---|
| `edge_id` | `fact_graph_edges` | `edge_id` | Degenerate key (grain) |
| `from_node_id` + `from_node_type` | `dim_customer` or `dim_product` → `fact_graph_edges` | `from_customer_sk` or `from_product_sk` | Role-playing; node type picks the dimension |
| `to_node_id` + `to_node_type` | `dim_customer` or `dim_product` → `fact_graph_edges` | `to_customer_sk` or `to_product_sk` | Role-playing; node type picks the dimension |
| `timestamp` | `dim_date` → `fact_graph_edges` | `date_key` → `event_date_key` | Derived `yyyymmdd` |
| `campaign` | `dim_campaign` → `fact_graph_edges` | `campaign_sk` | Lookup dimension |
| `customer_segment` | `dim_customer` and `fact_graph_edges` | `customer_segment` | Customer attribute, also kept on the fact |
| `product_id`, `category`, `price_bucket`, `unit_price_usd` | `dim_product` | `product_id`, `category`, `price_bucket`, `current_unit_price_usd` | Product attributes; price taken from the product's latest edge |
| `relationship`, `order_id`, `category`, `region`, `state`, `edge_strength`, `price_bucket`, `prior_interactions`, `dwell_seconds`, `unit_price_usd`, `quantity`, `returned_flag`, `auth_approved` | `fact_graph_edges` | same name | Measure / attribute, loaded as-is |
| `same_household` | none | none | Staged but not carried into the warehouse |

**Notes**
- `edges.product_id` feeds `dim_product` only; `fact_graph_edges` carries product keys solely through `from_node_id` / `to_node_id`.
- `same_household` is intentionally dropped; add a column to `fact_graph_edges` and the DDL if it is needed later.

## Key Fields, Relationships and ID Conforming Strategy

### Key fields

| Entity | Business key | Format | Appears in |
|---|---|---|---|
| Customer | `customer_id` / node id | `C#####` | orders, events, edges (`from_node_id` / `to_node_id` where type = Customer) |
| Product | `product_id` / node id | `P####` | events, edges (`product_id` and node ids where type = Product) |
| Order | `order_id` | `ORD######` | orders (grain), edges (`order_id`) |
| Event | `event_id` | `EVT######` | events (grain) |
| Edge | `edge_id` | `EDGE######` | edges (grain) |
| Session | `session_id` | `S##########` | events (degenerate dimension) |

### Relationships

- Customer 1:N orders, events and edges. Only 72 customers appear in both orders and events, so most customers are seen by a single source (7,057 in the union).
- Product 1:N events and edges. 378 products appear in both sources.
- Order 1:N edges. All 796 edge `order_id`s exist in orders.
- Edges are polymorphic: `from_node_type` / `to_node_type` say whether a node id is a customer or a product.

### Strategy for standardizing IDs

1. **Shared business keys.** All three systems already use the same prefixed format (`C…`, `P…`, `ORD…`), so IDs are loaded without remapping (only trimmed and upper-cased in Task 3.4).
2. **One conformed dimension per entity.** `dim_customer` is built from the union of customer ids across orders, events and edges. `dim_product` is built from events and edges. Each entity gets one surrogate key (`customer_sk`, `product_sk`).
3. **Surrogate keys in the facts.** Facts join to dimensions on the business key plus `is_current = TRUE`, so the SCD2 columns can later hold history.
4. **Node-type routing for edges.** `from_node_type` / `to_node_type` decide whether an edge end resolves to `dim_customer` or `dim_product`.
5. **Orphans kept.** LEFT JOINs keep fact rows whose id has no dimension match, so row counts match staging.
6. **Same type everywhere.** IDs are `VARCHAR(32)` in staging and the dimensions.

### Limitations

- IDs are validated against the patterns above for every row in Task 3.4; the pipeline stops if any ID does not match.
- IDs are only trimmed and upper-cased in Task 3.4; no other repair is attempted, so a malformed ID stops the load.

---
# Task 2: Design the Warehouse Schema

In this task, you will:
- Review the dimensional (star) schema design
- Understand staging tables, dimension tables, and fact tables
- Review distribution keys, sort keys, and encoding for optimization
- Document the purpose of each table

The DDL is defined in `project-ddl-long.md`. Review the schema design and understand how it supports analytics.

**Deliverables:**
- Understanding of the provided DDL structure
- Documentation of table purposes and query support

In [52]:
# Review the DDL file
print("="*60)
print("TASK 2: Warehouse Schema Design")
print("="*60)

print("\n📄 Reading DDL file:", DDL_MD_PATH)

# Read and parse the DDL file
with open(DDL_MD_PATH, 'r') as f:
    ddl_content = f.read()

sql_blocks = re.findall(r"```sql(.*?)```", ddl_content, flags=re.S)
print(f"Found {len(sql_blocks)} SQL blocks")

# Count and list the tables defined
table_names = re.findall(
    r"CREATE\s+TABLE\s+(?:IF\s+NOT\s+EXISTS\s+)?(\w+\.\w+)",
    "\n".join(sql_blocks),
    flags=re.I,
)

staging_tables = [t for t in table_names if t.startswith("stg.")]
dim_tables = [t for t in table_names if t.startswith("dw.dim_")]
fact_tables = [t for t in table_names if t.startswith("dw.fact_")]

for label, tables in [("Staging", staging_tables), ("Dimension", dim_tables), ("Fact", fact_tables)]:
    print(f"\n{label} tables ({len(tables)}):")
    for t in tables:
        print("  -", t)

TASK 2: Warehouse Schema Design

📄 Reading DDL file: ./project-ddl-long.md
Found 5 SQL blocks

Staging tables (3):
  - stg.orders_raw
  - stg.events_raw
  - stg.edges_raw

Dimension tables (12):
  - dw.dim_date
  - dw.dim_customer
  - dw.dim_product
  - dw.dim_campaign
  - dw.dim_channel
  - dw.dim_device
  - dw.dim_browser
  - dw.dim_os
  - dw.dim_referrer
  - dw.dim_shipping_method
  - dw.dim_payment_method
  - dw.dim_ab_variant

Fact tables (3):
  - dw.fact_orders
  - dw.fact_events
  - dw.fact_graph_edges


In [53]:
# Document the schema design
print("\n" + "="*60)
print("SCHEMA DESIGN DOCUMENTATION")
print("="*60)

schema_doc = """
## Staging Tables (stg schema)
Permissive landing zone: wide VARCHARs, minimal constraints, one table per source.
Staging decouples extraction from modelling, so raw data can be loaded, inspected and
reloaded without touching the warehouse tables.
- stg.orders_raw  : orders from PostgreSQL
- stg.events_raw  : clickstream events from Cassandra
- stg.edges_raw   : customer/product relationships from Neo4j

## Dimension Tables (dw schema)
Surrogate keys (IDENTITY) are used as the join keys from the facts.
- dim_customer, dim_product : SCD Type 2 ready (effective_from/to, is_current);
                              fact loads join on is_current = TRUE
- dim_date                  : yyyymmdd integer key; used for order, ship and event dates
- Small lookups             : dim_campaign, dim_channel, dim_device, dim_browser, dim_os,
                              dim_referrer, dim_shipping_method, dim_payment_method,
                              dim_ab_variant

## Fact Tables (dw schema)
- fact_orders      : one row per order (order_id); customer, order/ship date, channel,
                     device, browser, campaign, payment and shipping method keys
- fact_events      : one row per event (event_id); customer, product, date, channel,
                     device, browser, os, referrer, ab_variant keys; session_id and
                     event_type kept on the fact as degenerate dimensions
- fact_graph_edges : one row per graph edge (edge_id); from/to customer and product keys,
                     date and campaign keys

## Optimization Choices
- DISTKEY(customer_sk) on fact_orders and fact_events: rows for the same customer are
  collocated, so customer-centric joins and aggregations avoid data movement
- DISTKEY(to_product_sk) on fact_graph_edges: supports product-centric relationship analysis
- DISTKEY(customer_sk) on dim_customer (matches the fact join column, so customer joins are collocated); DISTSTYLE ALL on dim_product (joined from three fact columns)
- DISTSTYLE ALL on small lookup dimensions and dim_date: a full copy on every node, so
  joins to them are always local
- SORTKEY on the date key for all facts: time-range filters skip unneeded blocks;
  customer and product dimensions are sorted on their surrogate keys
- ENCODE zstd on most columns: good compression and less I/O
"""

print(schema_doc)


SCHEMA DESIGN DOCUMENTATION

## Staging Tables (stg schema)
Permissive landing zone: wide VARCHARs, minimal constraints, one table per source.
Staging decouples extraction from modelling, so raw data can be loaded, inspected and
reloaded without touching the warehouse tables.
- stg.orders_raw  : orders from PostgreSQL
- stg.events_raw  : clickstream events from Cassandra
- stg.edges_raw   : customer/product relationships from Neo4j

## Dimension Tables (dw schema)
Surrogate keys (IDENTITY) are used as the join keys from the facts.
- dim_customer, dim_product : SCD Type 2 ready (effective_from/to, is_current);
                              fact loads join on is_current = TRUE
- dim_date                  : yyyymmdd integer key; used for order, ship and event dates
- Small lookups             : dim_campaign, dim_channel, dim_device, dim_browser, dim_os,
                              dim_referrer, dim_shipping_method, dim_payment_method,
                              dim_ab_variant

## Fa

---
# Task 3: Extract and Transform the Source Data

In this task, you will:
- Load CSV data into PostgreSQL, Cassandra, and Neo4j (simulating production)
- Write extraction functions to query each source system
- Apply transformations to clean and conform the data
- Ensure transformed data matches staging table specifications

**Deliverables:**
- Working functions to connect, load, and extract from each source
- Transformed DataFrames ready for Redshift loading

## Task 3.1: Define Source System Functions

Implement the connection and data loading functions for each source system.

In [54]:
# ========= PostgreSQL Functions =========

def pg_connect():
    """Connect to PostgreSQL."""
    return psycopg2.connect(host=PG_HOST, port=PG_PORT, dbname=PG_DB, user=PG_USER, password=PG_PW)

PG_ORDERS_DDL = """
CREATE TABLE IF NOT EXISTS raw.orders (
  order_id VARCHAR(32) PRIMARY KEY, customer_id VARCHAR(32),
  order_datetime TIMESTAMP, ship_datetime TIMESTAMP,
  channel VARCHAR(32), device_type VARCHAR(16), browser VARCHAR(16),
  country VARCHAR(8), state VARCHAR(8), payment_method VARCHAR(16),
  campaign VARCHAR(32), primary_category VARCHAR(32), num_distinct_items INTEGER,
  subtotal_usd NUMERIC(12,2), discount_rate NUMERIC(5,3), discount_amount_usd NUMERIC(12,2),
  shipping_method VARCHAR(16), shipping_cost_usd NUMERIC(12,2),
  tax_rate NUMERIC(6,4), tax_amount_usd NUMERIC(12,2), order_total_usd NUMERIC(12,2),
  order_weight_kg NUMERIC(10,2), delivery_days INTEGER,
  on_time_delivery BOOLEAN, authorization_approved BOOLEAN, returned BOOLEAN
)"""

def _py(v):
    """Convert pandas/numpy scalars (incl. NaN/NaT) to plain Python for psycopg2."""
    if pd.isna(v):
        return None
    return v.item() if hasattr(v, "item") else (v.to_pydatetime() if hasattr(v, "to_pydatetime") else v)

def pg_load_orders(df: pd.DataFrame):
    """Create table and load orders data into PostgreSQL."""
    cols = [c for c, _ in ORDERS_COLSPEC]
    df = df.copy()
    for c, k in ORDERS_COLSPEC:
        if k == 'ts':
            df[c] = pd.to_datetime(df[c])
        elif k == 'b':
            df[c] = df[c].map({'True': True, 'False': False, True: True, False: False})
    rows = [tuple(_py(v) for v in r) for r in df[cols].itertuples(index=False, name=None)]
    with pg_connect() as conn, conn.cursor() as cur:
        cur.execute("CREATE SCHEMA IF NOT EXISTS raw;")
        cur.execute(PG_ORDERS_DDL)
        cur.execute("TRUNCATE raw.orders;")
        for i in range(0, len(rows), BATCH_SIZE):
            execute_values(cur, f"INSERT INTO raw.orders ({','.join(cols)}) VALUES %s", rows[i:i+BATCH_SIZE])
    print(f"   Loaded {len(rows)} orders into raw.orders")

def extract_from_pg() -> pd.DataFrame:
    """Extract orders from PostgreSQL."""
    engine = create_engine(f"postgresql+psycopg2://{PG_USER}:{PG_PW}@{PG_HOST}:{PG_PORT}/{PG_DB}")
    return pd.read_sql_query("SELECT * FROM raw.orders", engine)

print("PostgreSQL functions defined")


PostgreSQL functions defined


In [55]:
# ========= Cassandra Functions =========
from cassandra.concurrent import execute_concurrent_with_args

def _records(df: pd.DataFrame, colspec) -> List[tuple]:
    """DataFrame -> list of plain-Python tuples (None for NaN), typed per colspec."""
    df = df.copy()
    out = []
    conv = {
        's': lambda v: str(v),
        'ts': lambda v: pd.Timestamp(v).to_pydatetime(),
        'i': lambda v: int(v),
        'f': lambda v: float(v),
        'b': lambda v: v if isinstance(v, bool) else str(v).strip().lower() == 'true',
    }
    fns = [conv[k] for _, k in colspec]
    for row in df[[c for c, _ in colspec]].itertuples(index=False, name=None):
        out.append(tuple(None if pd.isna(v) else f(v) for v, f in zip(row, fns)))
    return out

CAS_TYPES = {'s': 'text', 'ts': 'timestamp', 'i': 'int', 'f': 'double', 'b': 'boolean'}

def cas_connect():
    """Connect to Cassandra and ensure keyspace exists."""
    auth = PlainTextAuthProvider(CAS_USER, CAS_PW) if CAS_USER else None
    cluster = Cluster(CAS_HOSTS, port=CAS_PORT, auth_provider=auth)
    session = cluster.connect()
    session.execute(f"CREATE KEYSPACE IF NOT EXISTS {CAS_KEYSPACE} "
                    "WITH replication = {'class': 'SimpleStrategy', 'replication_factor': 1}")
    session.set_keyspace(CAS_KEYSPACE)
    return session, cluster

def cas_load_events(df: pd.DataFrame):
    """Create table and load events data into Cassandra."""
    session, cluster = cas_connect()
    try:
        cols = ", ".join(f"{c} {CAS_TYPES[k]}" for c, k in EVENTS_COLSPEC)
        session.execute(f"CREATE TABLE IF NOT EXISTS events ({cols}, PRIMARY KEY (event_id))")
        session.execute("TRUNCATE events")
        names = [c for c, _ in EVENTS_COLSPEC]
        stmt = session.prepare(f"INSERT INTO events ({','.join(names)}) VALUES ({','.join('?' * len(names))})")
        results = execute_concurrent_with_args(session, stmt, _records(df, EVENTS_COLSPEC), concurrency=50)
        failed = [r for ok, r in results if not ok]
        if failed:
            raise RuntimeError(f"{len(failed)} inserts failed, e.g. {failed[0]}")
        print(f"   Loaded {len(results)} events into {CAS_KEYSPACE}.events")
    finally:
        cluster.shutdown()

def extract_from_cas() -> pd.DataFrame:
    """Extract events from Cassandra."""
    session, cluster = cas_connect()
    try:
        names = [c for c, _ in EVENTS_COLSPEC]
        rows = session.execute(f"SELECT {','.join(names)} FROM events")
        return pd.DataFrame(list(rows), columns=names)
    finally:
        cluster.shutdown()

print("Cassandra functions defined")


Cassandra functions defined


In [56]:
# ========= Neo4j Functions =========
NEO4J_REL_TYPES = {"VIEWED", "PURCHASED", "ADDED_TO_CART", "ALSO_BOUGHT_WITH", "REFERRED_FRIEND", "RETURNS"}
NEO4J_NODE_LABELS = {"Customer", "Product", "Order"}
EDGE_PROPS = [c for c, _ in EDGES_COLSPEC if c not in ("from_node_id", "from_node_type", "to_node_id", "to_node_type", "relationship")]

def neo4j_driver():
    """Connect to Neo4j."""
    return GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PW))

def neo4j_load_edges(df: pd.DataFrame):
    """Load edges into Neo4j as nodes and relationships."""
    # Labels and relationship types can't be Cypher parameters, so whitelist then interpolate.
    bad = (set(df.relationship) - NEO4J_REL_TYPES) | ((set(df.from_node_type) | set(df.to_node_type)) - NEO4J_NODE_LABELS)
    if bad:
        raise ValueError(f"Unexpected relationship/label values: {bad}")
    df = df.dropna(subset=["edge_id", "from_node_id", "to_node_id"])
    recs = _records(df, EDGES_COLSPEC)
    names = [c for c, _ in EDGES_COLSPEC]
    dicts = []
    for r in recs:
        d = dict(zip(names, r))
        d["timestamp"] = d["timestamp"].isoformat() if d["timestamp"] else None
        dicts.append(d)
    with neo4j_driver() as driver, driver.session() as s:
        s.run("MATCH (n) DETACH DELETE n").consume()
        for label in sorted(NEO4J_NODE_LABELS):
            s.run(f"CREATE CONSTRAINT {label.lower()}_id IF NOT EXISTS FOR (n:{label}) REQUIRE n.id IS UNIQUE").consume()
        n = 0
        groups = {}
        for d in dicts:
            groups.setdefault((d["from_node_type"], d["relationship"], d["to_node_type"]), []).append(d)
        for (fl, rel, tl), items in groups.items():
            q = (f"UNWIND $rows AS row "
                 f"MERGE (a:{fl} {{id: row.from_node_id}}) MERGE (b:{tl} {{id: row.to_node_id}}) "
                 f"MERGE (a)-[r:{rel} {{edge_id: row.edge_id}}]->(b) "
                 f"SET r += row.props")
            for i in range(0, len(items), BATCH_SIZE):
                batch = [{"from_node_id": x["from_node_id"], "to_node_id": x["to_node_id"], "edge_id": x["edge_id"],
                          "props": {k: x[k] for k in EDGE_PROPS if x[k] is not None}} for x in items[i:i+BATCH_SIZE]]
                s.run(q, rows=batch).consume()
            n += len(items)
    print(f"   Loaded {n} edges into Neo4j")

def extract_from_neo4j() -> pd.DataFrame:
    """Extract relationships from Neo4j as a tabular edge list."""
    q = ("MATCH (a)-[r]->(b) RETURN r.edge_id AS edge_id, a.id AS from_node_id, labels(a)[0] AS from_node_type, "
         "b.id AS to_node_id, labels(b)[0] AS to_node_type, type(r) AS relationship, properties(r) AS props")
    with neo4j_driver() as driver:
        recs, _, _ = driver.execute_query(q)
    rows = []
    for r in recs:
        d = {k: r[k] for k in ("edge_id", "from_node_id", "from_node_type", "to_node_id", "to_node_type", "relationship")}
        d.update(r["props"])
        rows.append(d)
    out = pd.DataFrame(rows).reindex(columns=[c for c, _ in EDGES_COLSPEC])
    out["timestamp"] = pd.to_datetime(out["timestamp"])
    return out.sort_values("edge_id").reset_index(drop=True)

print("Neo4j functions defined")


Neo4j functions defined


## Task 3.2: Load Source Systems

Load the CSV data into the operational databases (simulating production environment).

In [57]:
print("="*60)
print("TASK 3: Loading Source Systems")
print("="*60)

# TODO: Load data into each source system

# Load PostgreSQL
print("\n📦 Loading orders into PostgreSQL...")
pg_load_orders(orders_df)

# Load Cassandra
print("\n📦 Loading events into Cassandra...")
cas_load_events(events_df)

# Load Neo4j
print("\n📦 Loading edges into Neo4j...")
neo4j_load_edges(edges_df)

print("\n✅ All source systems loaded!")

TASK 3: Loading Source Systems

📦 Loading orders into PostgreSQL...
   Loaded 2500 orders into raw.orders

📦 Loading events into Cassandra...
   Loaded 2500 events into ecommerce.events

📦 Loading edges into Neo4j...
   Loaded 2500 edges into Neo4j

✅ All source systems loaded!


## Task 3.3: Extract and Transform

Extract data from source systems and transform for Redshift staging.

In [58]:
print("\n" + "="*60)
print("Extracting from Source Systems")
print("="*60)

# TODO: Extract data from each source system

print("\n📤 Extracting from PostgreSQL...")
orders_extracted = extract_from_pg()
print(f"   Extracted {len(orders_extracted)} orders")

print("\n📤 Extracting from Cassandra...")
events_extracted = extract_from_cas()
print(f"   Extracted {len(events_extracted)} events")

print("\n📤 Extracting from Neo4j...")
edges_extracted = extract_from_neo4j()
print(f"   Extracted {len(edges_extracted)} edges")

# TODO: Conform columns to staging specs
print("\n🔄 Conforming data to staging specifications...")
orders = orders_extracted[[c for c,_ in ORDERS_COLSPEC if c in orders_extracted.columns]].copy()
events = events_extracted[[c for c,_ in EVENTS_COLSPEC if c in events_extracted.columns]].copy()
edges = edges_extracted[[c for c,_ in EDGES_COLSPEC if c in edges_extracted.columns]].copy()

print("\n✅ Task 3 Complete - Data extracted and transformed!")


Extracting from Source Systems

📤 Extracting from PostgreSQL...
   Extracted 2500 orders

📤 Extracting from Cassandra...
   Extracted 2500 events

📤 Extracting from Neo4j...
   Extracted 2500 edges

🔄 Conforming data to staging specifications...

✅ Task 3 Complete - Data extracted and transformed!


In [59]:
# ========= Round-trip check: source CSV vs. data extracted back from each system
def roundtrip_check(name, csv_df, extracted, colspec, key):
    cols = [c for c, _ in colspec]
    a = csv_df[cols].copy(); b = extracted[cols].copy()
    for c, k in colspec:
        if k == 'ts':
            a[c] = pd.to_datetime(a[c]); b[c] = pd.to_datetime(b[c])
        elif k == 'b':
            a[c] = a[c].map(lambda v: v if pd.isna(v) else (v if isinstance(v, bool) else str(v) == 'True')).astype('boolean')
            b[c] = b[c].astype('boolean')
        elif k in ('i', 'f'):
            a[c] = pd.to_numeric(a[c]).astype('float64'); b[c] = pd.to_numeric(b[c]).astype('float64')
        else:
            a[c] = a[c].astype('object'); b[c] = b[c].astype('object')
    a = a.sort_values(key).reset_index(drop=True); b = b.sort_values(key).reset_index(drop=True)
    problems = []
    if len(a) != len(b): problems.append(f"row count {len(a)} vs {len(b)}")
    elif not a[key].equals(b[key]): problems.append("keys differ")
    else:
        for c, k in colspec:
            same = (a[c].eq(b[c]) | (a[c].isna() & b[c].isna())) if k != 'f' else \
                   ((a[c] - b[c]).abs().lt(1e-6) | (a[c].isna() & b[c].isna()))
            bad = int((~same.fillna(False)).sum())
            if bad: problems.append(f"{c}: {bad} mismatches")
    print(f"{'OK  ' if not problems else 'FAIL'} {name}: {len(a)} rows, {len(cols)} columns" + ("" if not problems else " -> " + "; ".join(problems)))
    return not problems

ok = all([
    roundtrip_check("orders (Postgres)", orders_df, orders_extracted, ORDERS_COLSPEC, "order_id"),
    roundtrip_check("events (Cassandra)", events_df, events_extracted, EVENTS_COLSPEC, "event_id"),
    roundtrip_check("edges (Neo4j)", edges_df, edges_extracted, EDGES_COLSPEC, "edge_id"),
])
assert ok, "Round-trip mismatch - see output above"


OK   orders (Postgres): 2500 rows, 26 columns
OK   events (Cassandra): 2500 rows, 24 columns
OK   edges (Neo4j): 2500 rows, 23 columns


## Task 3.4: Transform and Standardize

Conform IDs, clean categories and derive date keys on the extracted data before staging.

In [60]:
# ========= Task 3.4: Transform and standardize the extracted data
print("=" * 60)
print("Transform and Standardize")
print("=" * 60)

frames = {"orders": orders, "events": events, "edges": edges}
before = {name: df.copy() for name, df in frames.items()}

# 1) Conform IDs: trim, upper-case, then validate against the expected pattern
ID_PATTERNS = {
    "order_id": r"ORD\d{6}", "customer_id": r"C\d{5}", "event_id": r"EVT\d{6}",
    "session_id": r"S\d{10}", "product_id": r"P\d{4}", "edge_id": r"EDGE\d{6}",
    "from_node_id": r"C\d{5}|P\d{4}", "to_node_id": r"C\d{5}|P\d{4}",
}
print("\nConformed IDs (changed / not matching pattern):")
invalid_ids = 0
for name, df in frames.items():
    for col, pattern in ID_PATTERNS.items():
        if col not in df.columns:
            continue
        original = df[col].copy()
        df[col] = df[col].str.strip().str.upper()
        changed = int((original.fillna("") != df[col].fillna("")).sum())
        bad = int((df[col].notna() & ~df[col].fillna("").str.fullmatch(pattern)).sum())
        invalid_ids += bad
        print(f"   {name}.{col}: {changed} / {bad}")
assert invalid_ids == 0, "Some IDs do not match their expected pattern"

# 2) Clean categories: trim text, one country code per country, explicit value for 'no campaign'
CATEGORY_COLUMNS = {
    "orders": ["channel", "device_type", "browser", "country", "state", "payment_method",
               "campaign", "primary_category", "shipping_method"],
    "events": ["event_type", "device_type", "browser", "os", "referrer", "country", "state",
               "ab_variant", "payment_outcome", "category", "promo_code"],
    "edges":  ["from_node_type", "to_node_type", "relationship", "category", "customer_segment",
               "price_bucket", "region", "state", "campaign"],
}
for name, columns in CATEGORY_COLUMNS.items():
    for col in columns:
        frames[name][col] = frames[name][col].str.strip()

orders["country"] = orders["country"].replace({"UK": "GB"})   # events use the ISO code GB
orders["campaign"] = orders["campaign"].fillna("NONE")        # no campaign -> explicit NONE (as promo_code)
edges["campaign"] = edges["campaign"].fillna("NONE")

# 3) Derived fields: yyyymmdd date keys (extra columns, not part of the staging column specs)
def date_key(series):
    return pd.to_numeric(pd.to_datetime(series).dt.strftime("%Y%m%d"), errors="coerce").astype("Int64")

orders["order_date_key"] = date_key(orders["order_datetime"])
orders["ship_date_key"] = date_key(orders["ship_datetime"])
events["event_date_key"] = date_key(events["event_ts"])
edges["event_date_key"] = date_key(edges["timestamp"])

# 4) Show the visible changes
print("\nCleaned categories (before -> after):")
for name, col in [("orders", "country"), ("orders", "campaign"), ("edges", "campaign")]:
    print(f"\n   {name}.{col}")
    print("   before:", before[name][col].value_counts(dropna=False).to_dict())
    print("   after: ", frames[name][col].value_counts(dropna=False).to_dict())

print("\nDerived date keys (sample):")
display(orders[["order_id", "order_datetime", "order_date_key", "ship_datetime", "ship_date_key"]].head())
display(events[["event_id", "event_ts", "event_date_key"]].head())
display(edges[["edge_id", "timestamp", "event_date_key"]].head())

print("\n✅ Transform complete - IDs conformed, categories cleaned, date keys derived")

Transform and Standardize

Conformed IDs (changed / not matching pattern):
   orders.order_id: 0 / 0
   orders.customer_id: 0 / 0
   events.customer_id: 0 / 0
   events.event_id: 0 / 0
   events.session_id: 0 / 0
   events.product_id: 0 / 0
   edges.order_id: 0 / 0
   edges.product_id: 0 / 0
   edges.edge_id: 0 / 0
   edges.from_node_id: 0 / 0
   edges.to_node_id: 0 / 0

Cleaned categories (before -> after):

   orders.country
   before: {'US': 2174, 'CA': 68, 'FR': 67, 'AU': 67, 'UK': 66, 'DE': 58}
   after:  {'US': 2174, 'CA': 68, 'FR': 67, 'AU': 67, 'GB': 66, 'DE': 58}

   orders.campaign
   before: {'Holiday': 597, 'SpringSale': 447, nan: 409, 'BackToSchool': 292, 'Loyalty': 263, 'Clearance': 254, 'NewArrivals': 238}
   after:  {'Holiday': 597, 'SpringSale': 447, 'NONE': 409, 'BackToSchool': 292, 'Loyalty': 263, 'Clearance': 254, 'NewArrivals': 238}

   edges.campaign
   before: {'BackToSchool': 381, 'Clearance': 378, 'Loyalty': 363, nan: 352, 'NewArrivals': 350, 'SpringSale': 345,

,order_id,order_datetime,order_date_key,ship_datetime,ship_date_key
0,ORD100000,2024-06-12 02:14:21,20240612,2024-06-17 02:14:21,20240617
1,ORD100001,2024-08-08 00:16:41,20240808,2024-08-11 00:16:41,20240811
2,ORD100002,2024-09-11 06:59:18,20240911,2024-09-15 06:59:18,20240915
3,ORD100003,2025-03-13 15:54:14,20250313,2025-03-17 15:54:14,20250317
4,ORD100004,2025-02-28 00:20:21,20250228,2025-03-06 00:20:21,20250306


,event_id,event_ts,event_date_key
0,EVT200166,2025-05-20 12:16:03,20250520
1,EVT202405,2025-06-14 02:20:07,20250614
2,EVT201243,2024-06-11 19:30:25,20240611
3,EVT201787,2024-08-18 14:41:43,20240818
4,EVT202134,2024-10-30 21:08:51,20241030


,edge_id,timestamp,event_date_key
0,EDGE300000,2024-01-27 13:42:26,20240127
1,EDGE300001,2025-02-07 10:22:55,20250207
2,EDGE300002,2024-01-06 14:04:37,20240106
3,EDGE300003,2024-06-04 21:36:14,20240604
4,EDGE300004,2024-06-23 20:00:17,20240623



✅ Transform complete - IDs conformed, categories cleaned, date keys derived


## Task 3.5: Validate Against Column Specifications

Cast each column to the type its spec kind requires, then check names, order and types against `ORDERS_COLSPEC`, `EVENTS_COLSPEC` and `EDGES_COLSPEC`.

In [61]:
# ========= Task 3.5: Conform types and validate the transformed DataFrames against the column specs
print("=" * 60)
print("Validate Transformed DataFrames Against Column Specs")
print("=" * 60)

def to_bool(value):
    if pd.isna(value) or isinstance(value, (bool, np.bool_)):
        return value
    return str(value).strip().lower() == "true"

def cast_to_spec(df, colspec):
    """Cast each spec column to the pandas type that matches its kind."""
    for col, kind in colspec:
        if kind == "s":
            df[col] = df[col].astype(object)
        elif kind == "ts":
            df[col] = pd.to_datetime(df[col])
        elif kind == "i":
            df[col] = pd.to_numeric(df[col]).astype("Int64")
        elif kind == "f":
            df[col] = pd.to_numeric(df[col]).astype("float64")
        elif kind == "b":
            df[col] = df[col].map(to_bool).astype("boolean")
    return df

KIND_CHECKS = {
    "s":  lambda s: pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s),
    "ts": pd.api.types.is_datetime64_any_dtype,
    "i":  pd.api.types.is_integer_dtype,
    "f":  pd.api.types.is_float_dtype,
    "b":  pd.api.types.is_bool_dtype,
}

orders = cast_to_spec(orders, ORDERS_COLSPEC)
events = cast_to_spec(events, EVENTS_COLSPEC)
edges = cast_to_spec(edges, EDGES_COLSPEC)

report = []
for name, df, colspec in [("orders", orders, ORDERS_COLSPEC),
                          ("events", events, EVENTS_COLSPEC),
                          ("edges", edges, EDGES_COLSPEC)]:
    spec_columns = [c for c, _ in colspec]
    missing = [c for c in spec_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in spec_columns]
    wrong_type = [f"{c} ({kind})" for c, kind in colspec if c in df.columns and not KIND_CHECKS[kind](df[c])]
    in_order = [c for c in df.columns if c in spec_columns] == spec_columns
    ok = not missing and not wrong_type and in_order
    report.append({
        "source": name, "rows": len(df), "spec columns": len(spec_columns),
        "missing": missing or "-", "wrong type": wrong_type or "-",
        "same order": in_order, "extra (derived, not staged)": extra or "-",
        "matches spec": "PASS" if ok else "FAIL",
    })

display(pd.DataFrame(report))
assert all(r["matches spec"] == "PASS" for r in report), "Transformed DataFrames do not match the column specs"

print("\nColumn types after conforming (orders):")
display(orders[[c for c, _ in ORDERS_COLSPEC]].dtypes.rename("dtype").to_frame().T)
print("\n✅ Transformed DataFrames match the column specifications")

Validate Transformed DataFrames Against Column Specs


,source,rows,spec columns,missing,wrong type,same order,"extra (derived, not staged)",matches spec
0,orders,2500,26,-,-,True,"[order_date_key, ship_date_key]",PASS
1,events,2500,24,-,-,True,[event_date_key],PASS
2,edges,2500,23,-,-,True,[event_date_key],PASS



Column types after conforming (orders):


,order_id,customer_id,order_datetime,ship_datetime,channel,device_type,browser,country,state,payment_method,...,shipping_method,shipping_cost_usd,tax_rate,tax_amount_usd,order_total_usd,order_weight_kg,delivery_days,on_time_delivery,authorization_approved,returned
dtype,object,object,datetime64[us],datetime64[us],object,object,object,object,object,object,...,object,float64,float64,float64,float64,float64,Int64,boolean,boolean,boolean



✅ Transformed DataFrames match the column specifications


---
# Task 4: Load Data into Redshift

In this task, you will:
- Execute the DDL to create staging, dimension, and fact tables
- Load data into Redshift staging tables
- Populate dimension tables from staging data
- Populate fact tables with dimension key lookups
- Validate successful loading

**Deliverables:**
- Working Redshift connection and execution functions
- Loaded staging, dimension, and fact tables
- Row count validation

## Task 4.1: Define Redshift Functions

In [8]:
# ========= Redshift Functions =========

session_boto = boto3.Session(region_name=AWS_REGION)
rsd = session_boto.client("redshift-data", region_name=AWS_REGION)

def _rs_kwargs() -> Dict[str, Any]:
    """Build Redshift Data API connection parameters."""
    base = dict(Database=REDSHIFT_DATABASE)
    if REDSHIFT_WORKGROUP:
        base["WorkgroupName"] = REDSHIFT_WORKGROUP
        if REDSHIFT_SECRET_ARN:
            base["SecretArn"] = REDSHIFT_SECRET_ARN
    elif REDSHIFT_CLUSTER_IDENTIFIER and REDSHIFT_DB_USER:
        base["ClusterIdentifier"] = REDSHIFT_CLUSTER_IDENTIFIER
        base["DbUser"] = REDSHIFT_DB_USER
    else:
        raise RuntimeError("Configure Redshift serverless OR provisioned for Data API.")
    return base

def _sql_label(sql: str) -> str:
    """Short name for a statement, used in progress messages."""
    match = re.match(r"\s*((?:INSERT INTO|CREATE TABLE|DROP TABLE(?: IF EXISTS)?|CREATE MATERIALIZED VIEW|ANALYZE)\s+\S+)", sql, flags=re.I)
    return match.group(1) if match else " ".join(sql.split())[:50]

def rs_exec(sql: str, return_results=False, timeout_s=900, progress_after_s=3):
    """Execute SQL on Redshift via Data API; statements slower than progress_after_s report status and elapsed time."""
    import time
    stmt_id = rsd.execute_statement(Sql=sql, **_rs_kwargs())["Id"]
    label, start, delay, announced = _sql_label(sql), time.time(), 0.5, False
    while True:
        desc = rsd.describe_statement(Id=stmt_id)
        status = desc["Status"]
        elapsed = time.time() - start
        if status == "FINISHED":
            if announced:
                print(f"\r   {label}: finished in {elapsed:.0f}s" + " " * 30)
            break
        if status in ("FAILED", "ABORTED"):
            raise RuntimeError(f"Redshift statement {status}: {desc.get('Error', 'no error message')}\nSQL: {sql[:500]}")
        if time.time() - start > timeout_s:
            rsd.cancel_statement(Id=stmt_id)
            raise TimeoutError(f"Redshift statement timed out after {timeout_s}s\nSQL: {sql[:500]}")
        if elapsed >= progress_after_s:
            print(f"\r   {label}: {status.lower()} ... {elapsed:.0f}s", end="", flush=True)
            announced = True
        time.sleep(delay)
        delay = min(delay * 1.5, 5)
    if not return_results or not desc.get("HasResultSet"):
        return None
    rows, token = [], None
    while True:
        page = rsd.get_statement_result(Id=stmt_id, **({"NextToken": token} if token else {}))
        column_names = [column["name"] for column in page["ColumnMetadata"]]
        for record in page["Records"]:
            rows.append({name: (None if field.get("isNull") else next(iter(field.values()))) for name, field in zip(column_names, record)})
        token = page.get("NextToken")
        if not token:
            return rows

_SQL_FORMATTERS = {
    's': lambda v: "'" + str(v).replace("\\", "\\\\").replace("'", "''") + "'",
    'ts': lambda v: f"'{pd.Timestamp(v).strftime('%Y-%m-%d %H:%M:%S.%f')}'",
    'i': lambda v: str(int(v)),
    'f': lambda v: repr(float(v)),
    'b': lambda v: "TRUE" if (v if isinstance(v, bool) else str(v).strip().lower() == 'true') else "FALSE",
}
MAX_SQL_CHARS = 90_000  # Data API limits a statement to 100 KB

def _sql_literal(value, kind: str) -> str:
    """Format one value as a Redshift SQL literal (NULL for NaN/None)."""
    return "NULL" if pd.isna(value) else _SQL_FORMATTERS[kind](value)

def _batch_rows(row_sqls, prefix_chars: int):
    """Group row literals into batches capped by BATCH_SIZE rows and MAX_SQL_CHARS characters."""
    batch, chars = [], prefix_chars
    for row_sql in row_sqls:
        if batch and (len(batch) >= BATCH_SIZE or chars + len(row_sql) + 2 > MAX_SQL_CHARS):
            yield batch
            batch, chars = [], prefix_chars
        batch.append(row_sql)
        chars += len(row_sql) + 2
    if batch:
        yield batch

def rs_batch_insert(table: str, colspec: List[Tuple[str,str]], df: pd.DataFrame):
    """Load DataFrame into Redshift using batch INSERT statements."""
    column_names = [name for name, _ in colspec]
    kinds = [kind for _, kind in colspec]
    insert_prefix = f"INSERT INTO {table} ({', '.join(column_names)}) VALUES "
    row_sqls = ["(" + ", ".join(map(_sql_literal, row, kinds)) + ")"
                for row in df[column_names].itertuples(index=False, name=None)]
    batches = list(_batch_rows(row_sqls, len(insert_prefix)))
    for batch in (tqdm(batches, desc=table) if TQDM else batches):
        rs_exec(insert_prefix + ", ".join(batch))
    print(f"   Inserted {len(row_sqls)} rows into {table} in {len(batches)} batches")

print("Redshift functions defined")


Redshift functions defined


### Utility: Cancel Running Redshift Statements

`rs_cancel_active()` stops every active Redshift Data API statement (`SUBMITTED`, `PICKED` or `STARTED`) and prints its status, statement Id and the start of its SQL. Interrupting a notebook cell does not stop the statement in Redshift, so use this to clear a statement that is still running, for example before re-running the DDL cell.

- `rs_cancel_active()` cancels all active statements.
- `rs_cancel_active(cancel=False)` only lists them.

## Task 4.2: Execute DDL and Create Tables

In [88]:
print("="*60)
print("TASK 4: Loading Data into Redshift")
print("="*60)

print("\n📋 Step 1: Executing DDL to create tables...")

with open(DDL_MD_PATH, 'r') as f:
    md_content = f.read()

sql_blocks = re.findall(r"```sql(.*?)```", md_content, flags=re.DOTALL | re.IGNORECASE)

ddl_statements = []
for block in sql_blocks:
    for raw_statement in block.split(";"):
        code_lines = [line for line in raw_statement.splitlines() if not line.strip().startswith("--")]
        statement = "\n".join(code_lines).strip()
        if not statement or statement.upper().startswith("CREATE SCHEMA"):
            continue
        statement = re.sub(r'\bstg\.', 'public.stg_', statement)
        statement = re.sub(r'\bdw\.', 'public.dw_', statement)
        ddl_statements.append(statement)

for statement in (tqdm(ddl_statements, desc="DDL") if TQDM else ddl_statements):
    rs_exec(statement)
print(f"   Executed {len(ddl_statements)} DDL statements")


TASK 4: Loading Data into Redshift

📋 Step 1: Executing DDL to create tables...


DDL: 100%|██████████| 41/41 [01:25<00:00,  2.09s/it]

   Executed 41 DDL statements


## Task 4.3: Load Staging Tables

In [89]:
print("\n📦 Step 2: Loading staging tables...")

for name, table, colspec, frame in [
    ("orders", "public.stg_orders_raw", ORDERS_COLSPEC, orders),
    ("events", "public.stg_events_raw", EVENTS_COLSPEC, events),
    ("edges", "public.stg_edges_raw", EDGES_COLSPEC, edges),
]:
    print(f"\n  Loading {name} into {table}...")
    rs_batch_insert(table, colspec, frame)

print("\n✅ Staging tables loaded!")


📦 Step 2: Loading staging tables...

  Loading orders into public.stg_orders_raw...


public.stg_orders_raw: 100%|██████████| 8/8 [00:20<00:00,  2.59s/it]


   Inserted 2500 rows into public.stg_orders_raw in 8 batches

  Loading events into public.stg_events_raw...


public.stg_events_raw: 100%|██████████| 7/7 [00:18<00:00,  2.60s/it]


   Inserted 2500 rows into public.stg_events_raw in 7 batches

  Loading edges into public.stg_edges_raw...


public.stg_edges_raw: 100%|██████████| 7/7 [00:17<00:00,  2.56s/it]

   Inserted 2500 rows into public.stg_edges_raw in 7 batches

✅ Staging tables loaded!


## Task 4.4: Populate Dimension Tables

In [90]:
print("\n📊 Step 3: Populating dimension tables...")

# dim_date: every calendar date seen in orders, shipments, events and graph edges
rs_exec("""
    INSERT INTO public.dw_dim_date (date_key, date_actual, year, quarter, month, day, week_of_year, day_of_week, is_weekend)
    SELECT DISTINCT
        CAST(to_char(dt, 'YYYYMMDD') AS INTEGER) AS date_key,
        dt AS date_actual,
        EXTRACT(YEAR FROM dt)::SMALLINT,
        EXTRACT(QUARTER FROM dt)::SMALLINT,
        EXTRACT(MONTH FROM dt)::SMALLINT,
        EXTRACT(DAY FROM dt)::SMALLINT,
        EXTRACT(WEEK FROM dt)::SMALLINT,
        EXTRACT(DOW FROM dt)::SMALLINT,
        (EXTRACT(DOW FROM dt) IN (0,6))::BOOLEAN
    FROM (
        SELECT order_datetime::date AS dt FROM public.stg_orders_raw WHERE order_datetime IS NOT NULL
        UNION SELECT ship_datetime::date FROM public.stg_orders_raw WHERE ship_datetime IS NOT NULL
        UNION SELECT event_ts::date FROM public.stg_events_raw WHERE event_ts IS NOT NULL
        UNION SELECT "timestamp"::date FROM public.stg_edges_raw WHERE "timestamp" IS NOT NULL
    ) dates WHERE dt IS NOT NULL;
""")

# Single-column lookup dimensions: distinct non-null values from every staging column that carries them
LOOKUP_DIMENSIONS = {
    "dim_channel":         ("channel",         [("stg_orders_raw", "channel")]),
    "dim_device":          ("device_type",     [("stg_orders_raw", "device_type"), ("stg_events_raw", "device_type")]),
    "dim_browser":         ("browser",         [("stg_orders_raw", "browser"), ("stg_events_raw", "browser")]),
    "dim_shipping_method": ("shipping_method", [("stg_orders_raw", "shipping_method")]),
    "dim_payment_method":  ("payment_method",  [("stg_orders_raw", "payment_method")]),
    "dim_campaign":        ("campaign",        [("stg_orders_raw", "campaign"), ("stg_edges_raw", "campaign")]),
    "dim_os":              ("os",              [("stg_events_raw", "os")]),
    "dim_referrer":        ("referrer",        [("stg_events_raw", "referrer")]),
    "dim_ab_variant":      ("ab_variant",      [("stg_events_raw", "ab_variant")]),
}
for dimension, (column, sources) in LOOKUP_DIMENSIONS.items():
    distinct_values = " UNION ".join(
        f"SELECT DISTINCT {source_column} FROM public.{table} WHERE {source_column} IS NOT NULL"
        for table, source_column in sources
    )
    rs_exec(f"INSERT INTO public.dw_{dimension} ({column}) {distinct_values}")

# dim_customer: customers from orders, events and graph edges (the edge fact looks up both ends).
# Attributes are collapsed to one value per customer, since the SCD2 columns start with a single current row.
rs_exec("""
    INSERT INTO public.dw_dim_customer
        (customer_id, country, state, customer_segment, is_logged_in, effective_from, effective_to, is_current)
    SELECT ids.customer_id, location.country, location.state, segment.customer_segment, login.is_logged_in,
           GETDATE(), TIMESTAMP '9999-12-31 00:00:00', TRUE
    FROM (
        SELECT customer_id FROM public.stg_orders_raw
        UNION SELECT customer_id FROM public.stg_events_raw
        UNION SELECT from_node_id FROM public.stg_edges_raw WHERE from_node_type = 'Customer'
        UNION SELECT to_node_id FROM public.stg_edges_raw WHERE to_node_type = 'Customer'
    ) ids
    LEFT JOIN (
        SELECT customer_id, MAX(country) AS country, MAX(state) AS state
        FROM (
            SELECT customer_id, country, state FROM public.stg_orders_raw
            UNION ALL SELECT customer_id, country, state FROM public.stg_events_raw
        ) AS orders_and_events
        GROUP BY customer_id
    ) location ON location.customer_id = ids.customer_id
    LEFT JOIN (
        SELECT from_node_id AS customer_id, MAX(customer_segment) AS customer_segment
        FROM public.stg_edges_raw WHERE from_node_type = 'Customer'
        GROUP BY from_node_id
    ) segment ON segment.customer_id = ids.customer_id
    LEFT JOIN (
        SELECT customer_id, BOOL_OR(is_logged_in) AS is_logged_in
        FROM public.stg_events_raw GROUP BY customer_id
    ) login ON login.customer_id = ids.customer_id
    WHERE ids.customer_id IS NOT NULL;
""")

# dim_product: products from events and graph edges; price comes from each product's most recent edge
rs_exec("""
    INSERT INTO public.dw_dim_product
        (product_id, category, price_bucket, current_unit_price_usd, effective_from, effective_to, is_current)
    SELECT ids.product_id, categories.category, latest.price_bucket, latest.unit_price_usd,
           GETDATE(), TIMESTAMP '9999-12-31 00:00:00', TRUE
    FROM (
        SELECT product_id FROM public.stg_events_raw
        UNION SELECT product_id FROM public.stg_edges_raw
        UNION SELECT from_node_id FROM public.stg_edges_raw WHERE from_node_type = 'Product'
        UNION SELECT to_node_id FROM public.stg_edges_raw WHERE to_node_type = 'Product'
    ) ids
    LEFT JOIN (
        SELECT product_id, MAX(category) AS category
        FROM (
            SELECT product_id, category FROM public.stg_events_raw
            UNION ALL SELECT product_id, category FROM public.stg_edges_raw
        ) AS events_and_edges
        GROUP BY product_id
    ) categories ON categories.product_id = ids.product_id
    LEFT JOIN (
        SELECT product_id, price_bucket, unit_price_usd
        FROM (
            SELECT product_id, price_bucket, unit_price_usd,
                   ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY "timestamp" DESC) AS recency
            FROM public.stg_edges_raw
            WHERE product_id IS NOT NULL AND unit_price_usd IS NOT NULL
        ) AS ranked
        WHERE recency = 1
    ) latest ON latest.product_id = ids.product_id
    WHERE ids.product_id IS NOT NULL;
""")

for dimension in ["dim_date", "dim_customer", "dim_product", *LOOKUP_DIMENSIONS]:
    count = rs_exec(f"SELECT COUNT(*) AS n FROM public.dw_{dimension}", return_results=True)[0]["n"]
    print(f"   {dimension}: {count} rows")

print("\n✅ All dimension tables populated!")


📊 Step 3: Populating dimension tables...
   dim_date: 552 rows
   dim_customer: 7057 rows
   dim_product: 3360 rows
   dim_channel: 5 rows
   dim_device: 3 rows
   dim_browser: 5 rows
   dim_shipping_method: 3 rows
   dim_payment_method: 5 rows
   dim_campaign: 7 rows
   dim_os: 5 rows
   dim_referrer: 6 rows
   dim_ab_variant: 2 rows

✅ All dimension tables populated!


In [100]:
# Dimension duplicate check: each business key must appear once, otherwise the fact joins multiply rows
print("Dimension duplicate check (rows vs distinct keys):")
dimension_keys = {"dim_date": "date_key", "dim_customer": "customer_id", "dim_product": "product_id",
                  **{dim: column for dim, (column, _) in LOOKUP_DIMENSIONS.items()}}
duplicated = []
for dim, key in dimension_keys.items():
    r = rs_exec(f"SELECT COUNT(*) AS n, COUNT(DISTINCT {key}) AS d FROM public.dw_{dim}", return_results=True)[0]
    n, d = int(r["n"]), int(r["d"])
    print(f"   {'✓' if n == d else '✗'} {dim}: {n} rows, {d} distinct {key}")
    if n != d:
        duplicated.append(dim)
assert not duplicated, f"Duplicate rows in {duplicated}: re-run from the DDL cell before loading the facts"

Dimension duplicate check (rows vs distinct keys):
   ✓ dim_date: 552 rows, 552 distinct date_key
   ✓ dim_customer: 7057 rows, 7057 distinct customer_id
   ✓ dim_product: 3360 rows, 3360 distinct product_id
   ✓ dim_channel: 5 rows, 5 distinct channel
   ✓ dim_device: 3 rows, 3 distinct device_type
   ✓ dim_browser: 5 rows, 5 distinct browser
   ✓ dim_shipping_method: 3 rows, 3 distinct shipping_method
   ✓ dim_payment_method: 5 rows, 5 distinct payment_method
   ✓ dim_campaign: 7 rows, 7 distinct campaign
   ✓ dim_os: 5 rows, 5 distinct os
   ✓ dim_referrer: 6 rows, 6 distinct referrer
   ✓ dim_ab_variant: 2 rows, 2 distinct ab_variant


## Task 4.5: Populate Fact Tables

In [92]:
print("\n📊 Step 4: Populating fact tables...")

# Surrogate-key lookups use LEFT JOINs so rows with unmatched/NULL attributes still load; SCD2 dims join on is_current = TRUE.

# fact_orders: one row per order
rs_exec("""
    INSERT INTO public.dw_fact_orders (
        order_id, customer_sk, order_date_key, ship_date_key,
        channel_sk, device_sk, browser_sk, campaign_sk, payment_method_sk, shipping_method_sk,
        primary_category, num_distinct_items, subtotal_usd, discount_rate, discount_amount_usd,
        shipping_cost_usd, tax_rate, tax_amount_usd, order_total_usd, order_weight_kg,
        delivery_days, on_time_delivery, authorization_approved, returned
    )
    SELECT
        o.order_id,
        dc.customer_sk,
        CAST(to_char(o.order_datetime::date, 'YYYYMMDD') AS INTEGER),
        CAST(to_char(o.ship_datetime::date, 'YYYYMMDD') AS INTEGER),
        ch.channel_sk, dd.device_sk, br.browser_sk, ca.campaign_sk, pm.payment_method_sk, sm.shipping_method_sk,
        o.primary_category, o.num_distinct_items, o.subtotal_usd, o.discount_rate, o.discount_amount_usd,
        o.shipping_cost_usd, o.tax_rate, o.tax_amount_usd, o.order_total_usd, o.order_weight_kg,
        o.delivery_days, o.on_time_delivery, o.authorization_approved, o.returned
    FROM public.stg_orders_raw o
    LEFT JOIN public.dw_dim_customer dc ON dc.customer_id = o.customer_id AND dc.is_current = TRUE
    LEFT JOIN public.dw_dim_channel ch ON ch.channel = o.channel
    LEFT JOIN public.dw_dim_device dd ON dd.device_type = o.device_type
    LEFT JOIN public.dw_dim_browser br ON br.browser = o.browser
    LEFT JOIN public.dw_dim_campaign ca ON ca.campaign = o.campaign
    LEFT JOIN public.dw_dim_payment_method pm ON pm.payment_method = o.payment_method
    LEFT JOIN public.dw_dim_shipping_method sm ON sm.shipping_method = o.shipping_method;
""")

# fact_events: one row per event (events carry no channel, so fact_events has no channel_sk value)
rs_exec("""
    INSERT INTO public.dw_fact_events (
        event_id, customer_sk, product_sk, event_date_key, session_id, event_type,
        device_sk, browser_sk, os_sk, referrer_sk, ab_variant_sk,
        page_depth, latency_ms, dwell_seconds, cart_value_usd, discount_rate, fraud_score,
        payment_outcome, sequence_num, category, promo_code
    )
    SELECT
        e.event_id,
        dc.customer_sk,
        dp.product_sk,
        CAST(to_char(e.event_ts::date, 'YYYYMMDD') AS INTEGER),
        e.session_id, e.event_type,
        dd.device_sk, br.browser_sk, os.os_sk, rf.referrer_sk, ab.ab_variant_sk,
        e.page_depth, e.latency_ms, e.dwell_seconds, e.cart_value_usd, e.discount_rate, e.fraud_score,
        e.payment_outcome, e.sequence_num, e.category, e.promo_code
    FROM public.stg_events_raw e
    LEFT JOIN public.dw_dim_customer dc ON dc.customer_id = e.customer_id AND dc.is_current = TRUE
    LEFT JOIN public.dw_dim_product dp ON dp.product_id = e.product_id AND dp.is_current = TRUE
    LEFT JOIN public.dw_dim_device dd ON dd.device_type = e.device_type
    LEFT JOIN public.dw_dim_browser br ON br.browser = e.browser
    LEFT JOIN public.dw_dim_os os ON os.os = e.os
    LEFT JOIN public.dw_dim_referrer rf ON rf.referrer = e.referrer
    LEFT JOIN public.dw_dim_ab_variant ab ON ab.ab_variant = e.ab_variant;
""")

# fact_graph_edges: one row per edge; each end resolves to a customer or a product depending on its node type
rs_exec("""
    INSERT INTO public.dw_fact_graph_edges (
        edge_id, event_date_key, relationship,
        from_customer_sk, to_customer_sk, from_product_sk, to_product_sk,
        order_id, category, campaign_sk, customer_segment, region, state,
        edge_strength, price_bucket, prior_interactions, dwell_seconds,
        unit_price_usd, quantity, returned_flag, auth_approved
    )
    SELECT
        g.edge_id,
        CAST(to_char(g."timestamp"::date, 'YYYYMMDD') AS INTEGER),
        g.relationship,
        fc.customer_sk, tc.customer_sk, fp.product_sk, tp.product_sk,
        g.order_id, g.category, ca.campaign_sk, g.customer_segment, g.region, g.state,
        g.edge_strength, g.price_bucket, g.prior_interactions, g.dwell_seconds,
        g.unit_price_usd, g.quantity, g.returned_flag, g.auth_approved
    FROM public.stg_edges_raw g
    LEFT JOIN public.dw_dim_customer fc ON g.from_node_type = 'Customer' AND fc.customer_id = g.from_node_id AND fc.is_current = TRUE
    LEFT JOIN public.dw_dim_customer tc ON g.to_node_type = 'Customer' AND tc.customer_id = g.to_node_id AND tc.is_current = TRUE
    LEFT JOIN public.dw_dim_product fp ON g.from_node_type = 'Product' AND fp.product_id = g.from_node_id AND fp.is_current = TRUE
    LEFT JOIN public.dw_dim_product tp ON g.to_node_type = 'Product' AND tp.product_id = g.to_node_id AND tp.is_current = TRUE
    LEFT JOIN public.dw_dim_campaign ca ON ca.campaign = g.campaign;
""")

for fact in ["fact_orders", "fact_events", "fact_graph_edges"]:
    count = rs_exec(f"SELECT COUNT(*) AS n FROM public.dw_{fact}", return_results=True)[0]["n"]
    print(f"   {fact}: {count} rows")

print("\n✅ Task 4 Complete - All data loaded into Redshift!")


📊 Step 4: Populating fact tables...
   INSERT INTO public.dw_fact_orders: finished in 6s                              
   INSERT INTO public.dw_fact_events: finished in 6s                              
   fact_orders: 2500 rows
   fact_events: 2500 rows
   fact_graph_edges: 2500 rows

✅ Task 4 Complete - All data loaded into Redshift!


---
# Task 5: Optimize Performance and Build OLAP Structures

In this task, you will:
- Verify distribution styles and sort keys are applied
- Run ANALYZE to update statistics
- Create materialized views for common queries

**Deliverables:**
- At least one materialized view for common analytics
- ANALYZE run on key tables

## Distribution and Sort Key Strategy

Choices for every warehouse table, as defined in `project-ddl-long.md`.

| Table | Distribution | Sort key | Justification |
|---|---|---|---|
| `fact_orders` | `DISTKEY(customer_sk)` | `SORTKEY(order_date_key)` | Customer-centric fact. Distributing on the same column as `dim_customer` collocates customer joins and aggregations. Sorting on the order date lets date-range filters (such as daily revenue) skip blocks. |
| `fact_events` | `DISTKEY(customer_sk)` | `SORTKEY(event_date_key)` | Same customer key as `fact_orders`, so event-to-order and event-to-customer joins stay on one node. Date sort key serves time-range clickstream queries. |
| `fact_graph_edges` | `DISTKEY(to_product_sk)` | `SORTKEY(event_date_key)` | Relationship analysis is product-centric, so edges are grouped by the target product. Trade-off: edges whose target is a customer have a NULL `to_product_sk` and land on one slice, which is acceptable at this volume. Date sort key for time filters. |
| `dim_date` | `DISTSTYLE ALL` | `SORTKEY(date_key)` | 552 rows, joined by every fact. A full copy on each node keeps date joins local; sorted on the `yyyymmdd` key for range lookups. |
| `dim_customer` | `DISTKEY(customer_sk)` | `SORTKEY(customer_sk)` | 7,057 rows. Distributed on the surrogate key the facts join on, so `fact_orders` and `fact_events` joins are collocated. Sorted on the same key for merge joins. |
| `dim_product` | `DISTSTYLE ALL` | `SORTKEY(product_sk)` | 3,360 rows, joined from three different fact columns (`product_sk`, `from_product_sk`, `to_product_sk`), so no single `DISTKEY` can collocate all of them. A full copy per node keeps every join local. |
| `dim_campaign` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (7 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_channel` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (5 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_device` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (3 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_browser` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (5 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_os` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (5 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_referrer` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (6 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_shipping_method` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (3 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_payment_method` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (5 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |
| `dim_ab_variant` | `DISTSTYLE ALL` | none (AUTO) | Tiny lookup (2 values). Broadcast to every node so joins need no shuffling; a sort key would add nothing. |

**Principles**
- Facts are distributed on the column they are most often joined or grouped by, and sorted on a date key because most analytical queries filter by time.
- Dimensions that are small, or joined from several fact columns, use `DISTSTYLE ALL` so no join needs to move data.
- Only `dim_customer` is large and single-keyed enough to distribute on its surrogate key, matching the customer-centric facts.

In [94]:
print("="*60)
print("TASK 5: Optimize Performance")
print("="*60)

print("\n📊 Creating materialized view for daily revenue...")

# Redshift doesn't support "IF NOT EXISTS" for materialized views, so DROP + CREATE
rs_exec("DROP MATERIALIZED VIEW IF EXISTS public.dw_mv_daily_revenue;")
rs_exec("""
    CREATE MATERIALIZED VIEW public.dw_mv_daily_revenue AS
    SELECT
        order_date_key,
        COUNT(*) AS orders,
        SUM(order_total_usd) AS revenue_usd,
        AVG(order_total_usd) AS avg_order_value
    FROM public.dw_fact_orders
    GROUP BY order_date_key;
""")
count = rs_exec("SELECT COUNT(*) AS n FROM public.dw_mv_daily_revenue", return_results=True)[0]["n"]
print(f"   dw_mv_daily_revenue: {count} rows")

TASK 5: Optimize Performance

📊 Creating materialized view for daily revenue...
   dw_mv_daily_revenue: 542 rows


In [95]:
print("\n📊 Running ANALYZE on tables...")

for table in ['dw_fact_orders', 'dw_fact_events', 'dw_fact_graph_edges',
              'dw_dim_customer', 'dw_dim_product', 'dw_dim_date']:
    rs_exec(f"ANALYZE public.{table};")
    print(f"  ✓ ANALYZE complete: {table}")

print("\n✅ Task 5 Complete - Performance optimization done!")


📊 Running ANALYZE on tables...
  ✓ ANALYZE complete: dw_fact_orders
  ✓ ANALYZE complete: dw_fact_events
  ✓ ANALYZE complete: dw_fact_graph_edges
  ✓ ANALYZE complete: dw_dim_customer
  ✓ ANALYZE complete: dw_dim_product
  ✓ ANALYZE complete: dw_dim_date

✅ Task 5 Complete - Performance optimization done!


In [10]:
# ========= Verify the distribution and sort keys were applied in Redshift
print("Distribution and sort keys applied in Redshift (pg_class / pg_table_def):")

# (distribution, first sort key) expected from the DDL; None = no explicit sort key
EXPECTED_KEYS = {
    "dw_fact_orders": ("customer_sk", "order_date_key"),
    "dw_fact_events": ("customer_sk", "event_date_key"),
    "dw_fact_graph_edges": ("to_product_sk", "event_date_key"),
    "dw_dim_date": ("ALL", "date_key"),
    "dw_dim_customer": ("customer_sk", "customer_sk"),
    "dw_dim_product": ("ALL", "product_sk"),
}
# svv_table_info needs elevated rights; pg_class / pg_table_def are readable by normal users
dist_info = pd.DataFrame(rs_exec("""
    SELECT c.relname AS table_name,
           CASE c.reldiststyle WHEN 0 THEN 'EVEN' WHEN 1 THEN 'KEY' WHEN 8 THEN 'ALL'
                WHEN 10 THEN 'AUTO(ALL)' WHEN 11 THEN 'AUTO(EVEN)' WHEN 12 THEN 'AUTO(KEY)'
                ELSE 'OTHER' END AS diststyle
    FROM pg_class c JOIN pg_namespace n ON n.oid = c.relnamespace
    WHERE n.nspname = 'public' AND LEFT(c.relname, 3) = 'dw_' AND c.relkind = 'r'
""", return_results=True))
key_info = pd.DataFrame(rs_exec("""
    SELECT tablename AS table_name,
           MAX(CASE WHEN distkey THEN "column" END) AS distkey,
           MAX(CASE WHEN sortkey = 1 THEN "column" END) AS sortkey1
    FROM pg_table_def
    WHERE schemaname = 'public' AND LEFT(tablename, 3) = 'dw_'
    GROUP BY tablename
""", return_results=True))
table_info = dist_info.merge(key_info, on="table_name", how="left").sort_values("table_name")
table_info["tbl_rows"] = [
    int(rs_exec(f"SELECT COUNT(*) AS n FROM public.{t}", return_results=True)[0]["n"])
    for t in table_info["table_name"]
]

def keys_applied(row):
    expected_dist, expected_sort = EXPECTED_KEYS.get(row.table_name, ("ALL", None))
    dist_ok = expected_dist.lower() in (str(row.diststyle) + " " + str(row.distkey)).lower()
    sort_ok = expected_sort is None or str(row.sortkey1).lower() == expected_sort
    return "✓" if dist_ok and sort_ok else "✗"

table_info["applied"] = table_info.apply(keys_applied, axis=1)
display(table_info)

Distribution and sort keys applied in Redshift (pg_class / pg_table_def):


,table_name,diststyle,distkey,sortkey1,tbl_rows,applied
14,dw_dim_ab_variant,ALL,NaN,NaN,2,✓
9,dw_dim_browser,ALL,NaN,NaN,5,✓
6,dw_dim_campaign,ALL,NaN,NaN,7,✓
7,dw_dim_channel,ALL,NaN,NaN,5,✓
3,dw_dim_customer,KEY,customer_sk,customer_sk,7057,✓
5,dw_dim_date,ALL,NaN,date_key,552,✓
8,dw_dim_device,ALL,NaN,NaN,3,✓
10,dw_dim_os,ALL,NaN,NaN,5,✓
13,dw_dim_payment_method,ALL,NaN,NaN,5,✓
4,dw_dim_product,ALL,NaN,product_sk,3360,✓


In [11]:
# ========= Measure the effect of the materialized view (server-side query time)
import time
import uuid

RUNS = 5
tag = uuid.uuid4().hex[:8]
QUERIES = {
    "fact table": """SELECT order_date_key, COUNT(*) AS orders, SUM(order_total_usd) AS revenue_usd,
                            AVG(order_total_usd) AS avg_order_value
                     FROM public.dw_fact_orders GROUP BY order_date_key""",
    "materialized view": """SELECT order_date_key, orders, revenue_usd, avg_order_value
                            FROM public.dw_mv_daily_revenue""",
}

# Each run carries a unique comment so Redshift's result cache cannot answer it
for label, sql in QUERIES.items():
    for run in range(RUNS):
        rs_exec(f"/* perf_{tag}_{label.split()[0]}_{run} */ {sql}", return_results=True)

# sys_query_history lags a few seconds behind, so poll until every run is recorded
for attempt in range(10):
    rows = rs_exec(f"""
        SELECT CASE WHEN query_text LIKE '%perf_{tag}_fact_%' THEN 'fact table' ELSE 'materialized view' END AS source,
               COUNT(*) AS runs, AVG(elapsed_time) / 1000.0 AS avg_ms, MIN(elapsed_time) / 1000.0 AS min_ms
        FROM sys_query_history
        WHERE query_text LIKE '%perf_{tag}_%' AND query_text NOT LIKE '%sys_query_history%'
        GROUP BY 1
    """, return_results=True)
    if sum(int(r["runs"]) for r in rows) >= 2 * RUNS:
        break
    time.sleep(5)

timings = pd.DataFrame(rows).astype({"runs": int, "avg_ms": float, "min_ms": float}).round(1)
display(timings)
by_source = timings.set_index("source")["avg_ms"]
if {"fact table", "materialized view"} <= set(by_source.index) and by_source["materialized view"] > 0:
    print(f"Average: fact table {by_source['fact table']} ms vs materialized view {by_source['materialized view']} ms "
          f"({by_source['fact table'] / by_source['materialized view']:.1f}x)")

,source,runs,avg_ms,min_ms
0,fact table,5,55.5,3.7
1,materialized view,5,44.1,3.5


Average: fact table 55.5 ms vs materialized view 44.1 ms (1.3x)


---
# Task 6: Validate and Report Your Results

In this task, you will:
- Run data quality checks
- Execute sample analytical queries
- Generate the final report

**Deliverables:**
- Data quality checks (row counts, null checks)
- Sample analytical query results
- Final report with schema diagram and design rationale

In [12]:
print("="*60)
print("TASK 6: Validation and Reporting")
print("="*60)

STAGING_TABLES = ["stg_orders_raw", "stg_events_raw", "stg_edges_raw"]
FACT_TABLES = ["dw_fact_orders", "dw_fact_events", "dw_fact_graph_edges"]
DIMENSION_TABLES = [
    "dw_dim_date", "dw_dim_customer", "dw_dim_product", "dw_dim_campaign", "dw_dim_channel",
    "dw_dim_device", "dw_dim_browser", "dw_dim_os", "dw_dim_referrer",
    "dw_dim_shipping_method", "dw_dim_payment_method", "dw_dim_ab_variant",
]

print("\n📊 Row Counts:")
print("-" * 40)

row_count_query = " UNION ALL ".join(
    f"SELECT '{table}' AS table_name, COUNT(*) AS n FROM public.{table}"
    for table in [*STAGING_TABLES, *FACT_TABLES, *DIMENSION_TABLES]
)
row_counts = {r["table_name"]: int(r["n"]) for r in rs_exec(row_count_query, return_results=True)}

for heading, tables in [("Staging", STAGING_TABLES), ("Facts", FACT_TABLES), ("Dimensions", DIMENSION_TABLES)]:
    print(f"\n  {heading}")
    for table in tables:
        print(f"    {table}: {row_counts[table]:,}")

print("\n📊 Data Quality Checks:")
print("-" * 40)

# Each fact table should hold one row per staging row (the grain is preserved by the LEFT JOINs)
for fact, staging in [("dw_fact_orders", "stg_orders_raw"), ("dw_fact_events", "stg_events_raw"), ("dw_fact_graph_edges", "stg_edges_raw")]:
    status = "✓" if row_counts[fact] == row_counts[staging] else "✗"
    print(f"  {status} {fact} ({row_counts[fact]:,}) vs {staging} ({row_counts[staging]:,})")

# Unresolved surrogate keys: every row should have these populated
null_checks = [
    ("dw_fact_orders", "customer_sk IS NULL"),
    ("dw_fact_orders", "order_date_key IS NULL"),
    ("dw_fact_events", "customer_sk IS NULL"),
    ("dw_fact_events", "event_date_key IS NULL"),
    ("dw_fact_graph_edges", "from_customer_sk IS NULL AND from_product_sk IS NULL"),
    ("dw_fact_graph_edges", "to_customer_sk IS NULL AND to_product_sk IS NULL"),
]
for table, condition in null_checks:
    n = int(rs_exec(f"SELECT COUNT(*) AS n FROM public.{table} WHERE {condition}", return_results=True)[0]["n"])
    print(f"  {'✓' if n == 0 else '✗'} {table} WHERE {condition}: {n:,}")

# Duplicate business keys: each fact has one row per order / event / edge
print("\n📊 Duplicate Checks:")
for table, key in [("dw_fact_orders", "order_id"), ("dw_fact_events", "event_id"), ("dw_fact_graph_edges", "edge_id")]:
    n = int(rs_exec(
        f"SELECT COUNT(*) AS n FROM (SELECT {key} FROM public.{table} GROUP BY {key} HAVING COUNT(*) > 1) d",
        return_results=True)[0]["n"])
    print(f"  {'✓' if n == 0 else '✗'} {table}: duplicate {key} values: {n:,}")

# Orphan keys: every non-NULL foreign key must exist in its dimension
print("\n📊 Orphan Key Checks:")
orphan_checks = [
    ("dw_fact_orders", "customer_sk", "dw_dim_customer", "customer_sk"),
    ("dw_fact_orders", "order_date_key", "dw_dim_date", "date_key"),
    ("dw_fact_orders", "ship_date_key", "dw_dim_date", "date_key"),
    ("dw_fact_events", "customer_sk", "dw_dim_customer", "customer_sk"),
    ("dw_fact_events", "product_sk", "dw_dim_product", "product_sk"),
    ("dw_fact_events", "event_date_key", "dw_dim_date", "date_key"),
    ("dw_fact_graph_edges", "from_product_sk", "dw_dim_product", "product_sk"),
    ("dw_fact_graph_edges", "to_product_sk", "dw_dim_product", "product_sk"),
    ("dw_fact_graph_edges", "from_customer_sk", "dw_dim_customer", "customer_sk"),
    ("dw_fact_graph_edges", "to_customer_sk", "dw_dim_customer", "customer_sk"),
]
for fact, fk, dim, pk in orphan_checks:
    n = int(rs_exec(
        f"SELECT COUNT(*) AS n FROM public.{fact} f LEFT JOIN public.{dim} d ON f.{fk} = d.{pk} "
        f"WHERE f.{fk} IS NOT NULL AND d.{pk} IS NULL", return_results=True)[0]["n"])
    print(f"  {'✓' if n == 0 else '✗'} {fact}.{fk} -> {dim}.{pk}: {n:,} orphans")

TASK 6: Validation and Reporting

📊 Row Counts:
----------------------------------------

  Staging
    stg_orders_raw: 2,500
    stg_events_raw: 2,500
    stg_edges_raw: 2,500

  Facts
    dw_fact_orders: 2,500
    dw_fact_events: 2,500
    dw_fact_graph_edges: 2,500

  Dimensions
    dw_dim_date: 552
    dw_dim_customer: 7,057
    dw_dim_product: 3,360
    dw_dim_campaign: 7
    dw_dim_channel: 5
    dw_dim_device: 3
    dw_dim_browser: 5
    dw_dim_os: 5
    dw_dim_referrer: 6
    dw_dim_shipping_method: 3
    dw_dim_payment_method: 5
    dw_dim_ab_variant: 2

📊 Data Quality Checks:
----------------------------------------
  ✓ dw_fact_orders (2,500) vs stg_orders_raw (2,500)
  ✓ dw_fact_events (2,500) vs stg_events_raw (2,500)
  ✓ dw_fact_graph_edges (2,500) vs stg_edges_raw (2,500)
  ✓ dw_fact_orders WHERE customer_sk IS NULL: 0
  ✓ dw_fact_orders WHERE order_date_key IS NULL: 0
  ✓ dw_fact_events WHERE customer_sk IS NULL: 0
  ✓ dw_fact_events WHERE event_date_key IS NULL: 0
  ✓ d

In [98]:
print("\n📊 Sample Analytics - Daily Revenue:")
print("-" * 40)

# Query the materialized view, joined to dim_date for readable dates
daily_rev = rs_exec("""
    SELECT d.date_actual, mv.revenue_usd, mv.orders, mv.avg_order_value
    FROM public.dw_mv_daily_revenue mv
    JOIN public.dw_dim_date d ON d.date_key = mv.order_date_key
    ORDER BY d.date_actual
    LIMIT 10;
""", return_results=True)
daily_rev_df = pd.DataFrame(daily_rev)
display(daily_rev_df)

print("\n📊 Sample Analytics - Revenue by Channel:")
print("-" * 40)

channel_rev = rs_exec("""
    SELECT ch.channel, COUNT(*) AS orders, SUM(f.order_total_usd) AS revenue_usd
    FROM public.dw_fact_orders f
    JOIN public.dw_dim_channel ch ON ch.channel_sk = f.channel_sk
    GROUP BY ch.channel
    ORDER BY revenue_usd DESC;
""", return_results=True)
channel_rev_df = pd.DataFrame(channel_rev)
display(channel_rev_df)

print("\n📊 Sample Analytics - Events by Type:")
print("-" * 40)

event_types = rs_exec("""
    SELECT event_type, COUNT(*) AS events, COUNT(DISTINCT session_id) AS sessions
    FROM public.dw_fact_events
    GROUP BY event_type
    ORDER BY events DESC;
""", return_results=True)
event_types_df = pd.DataFrame(event_types)
display(event_types_df)


📊 Sample Analytics - Daily Revenue:
----------------------------------------


,date_actual,revenue_usd,orders,avg_order_value
0,2024-01-01,4086.97,6,681.16
1,2024-01-02,2150.01,5,430.00
2,2024-01-03,681.74,6,113.62
3,2024-01-04,731.67,3,243.89
4,2024-01-05,1302.17,4,325.54
5,2024-01-06,4421.65,9,491.29
6,2024-01-07,2293.13,5,458.62
7,2024-01-09,1712.60,6,285.43
8,2024-01-10,2472.00,8,309.00
9,2024-01-11,4127.43,6,687.90



📊 Sample Analytics - Revenue by Channel:
----------------------------------------


,channel,orders,revenue_usd
0,android_app,535,224060.04
1,ios_app,498,221419.83
2,web,500,205995.94
3,marketplace,490,199147.95
4,mobile_web,477,188793.93



📊 Sample Analytics - Events by Type:
----------------------------------------


,event_type,events,sessions
0,product_view,696,696
1,page_view,641,641
2,add_to_cart,480,480
3,checkout_start,259,259
4,payment_attempt,218,218
5,purchase,155,155
6,return_initiated,51,51


In [99]:
print("\n📄 Generating Final Report...")

def counts_table(tables):
    rows = "\n".join(f"| `{table}` | {row_counts[table]:,} |" for table in tables)
    return f"| Table | Rows |\n|---|---:|\n{rows}"

def df_table(df):
    try:
        return df.to_markdown(index=False)  # needs the optional tabulate package
    except ImportError:
        return "```\n" + df.to_string(index=False) + "\n```"

# Embed the ER diagram from the project file when it is available
diagram_path = os.path.join(BASE_DIR, "project-mermaid-diagram.md")
if os.path.exists(diagram_path):
    diagram_source = "\n".join(l for l in open(diagram_path).read().splitlines() if not l.startswith("#"))
    schema_diagram = "```mermaid\n" + diagram_source.strip() + "\n```"
else:
    schema_diagram = "_See project-mermaid-diagram.md_"

report_content = f"""# Data Warehouse Build Report

Generated: {datetime.utcnow().isoformat()}Z

## Schema Overview

{schema_diagram}

### Staging Tables
{counts_table(STAGING_TABLES)}

### Fact Tables
{counts_table(FACT_TABLES)}

### Dimension Tables
{counts_table(DIMENSION_TABLES)}

## Design Rationale

1. **Star schema.** Three facts at different grains (orders, events, graph edges) share conformed dimensions
   (customer, product, date, campaign, device, browser). Analysts can slice any fact by the same attributes,
   and joins stay one hop from fact to dimension.
2. **Distribution keys.** `fact_orders` and `fact_events` use `DISTKEY(customer_sk)` so customer-centric joins and
   aggregations are collocated. `fact_graph_edges` uses `DISTKEY(to_product_sk)` for product-centric analysis.
   Small lookup dimensions use `DISTSTYLE ALL` so they are broadcast to every node and joins need no shuffling.
3. **Sort keys.** Every fact is sorted on its date key (`order_date_key`, `event_date_key`), so time-range filters
   skip blocks via zone maps. `dim_date` is sorted on `date_key`; customer and product dimensions on their surrogate keys.
4. **SCD2-ready dimensions.** `dim_customer` and `dim_product` carry `effective_from`, `effective_to` and `is_current`;
   fact loads join on `is_current = TRUE`.
5. **Materialized view.** `dw_mv_daily_revenue` pre-aggregates orders, revenue and average order value per day,
   so dashboards read a small table instead of scanning `fact_orders`.

## Analytics Capabilities

- Revenue, order volume and average order value over time (daily, by weekday, by quarter via `dim_date`)
- Revenue and orders by channel, campaign, device, browser, payment and shipping method
- Delivery performance and returns (`on_time_delivery`, `delivery_days`, `returned`)
- Clickstream and funnel analysis by event type, session, device, referrer and A/B variant
- Product relationship analysis from graph edges (customer-to-product and product-to-product)

## Sample Query Results

### Daily revenue (first 10 days)
{df_table(daily_rev_df)}

### Revenue by channel
{df_table(channel_rev_df)}

### Events by type
{df_table(event_types_df)}
"""

# Save report
report_path = os.path.join(BASE_DIR, "warehouse_report.md")
with open(report_path, "w") as f:
    f.write(report_content)

print(f"\n✅ Report saved to: {report_path}")
print("\n" + "="*60)
print("ALL TASKS COMPLETED! ✅")
print("="*60)


📄 Generating Final Report...

✅ Report saved to: ./warehouse_report.md

ALL TASKS COMPLETED! ✅


/var/folders/sh/tl74dmfn2d38g0d22l4xvvhw0000gn/T/ipykernel_5181/2455224404.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  Generated: {datetime.utcnow().isoformat()}Z


# Redshift Helper Functions I used when things went wrong

### Utility: Cancel Running Redshift Statements

`rs_cancel_active()` stops every active Redshift Data API statement (`SUBMITTED`, `PICKED` or `STARTED`) and prints its status, statement Id and the start of its SQL. Interrupting a notebook cell does not stop the statement in Redshift, so use this to clear a statement that is still running, for example before re-running the DDL cell.

- `rs_cancel_active()` cancels all active statements.
- `rs_cancel_active(cancel=False)` only lists them.

In [101]:
def rs_cancel_active(cancel: bool = True):
    """Cancel every active Redshift Data API statement (cancel=False only lists them)."""
    active = []
    for status in ("SUBMITTED", "PICKED", "STARTED"):
        token = None
        while True:
            page = rsd.list_statements(Status=status, MaxResults=100, **({"NextToken": token} if token else {}))
            active += page["Statements"]
            token = page.get("NextToken")
            if not token:
                break
    if not active:
        print("No active statements")
    for s in sorted(active, key=lambda s: s["CreatedAt"]):
        if cancel:
            rsd.cancel_statement(Id=s["Id"])
        sql = " ".join(s.get("QueryString", "").split())[:80]
        print(f"{'cancelled' if cancel else s['Status'].lower():<10} {s['Id']}  {sql}")

print("rs_cancel_active() defined")

rs_cancel_active() defined
